Phase 4 as “Independent Demand Signal & Distribution Gap.”

In [98]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
OUTPUT_DATA = PROJECT_ROOT / "data" / "outputs"

OUTPUT_DATA.mkdir(parents=True, exist_ok=True)

print("Notebook:", NOTEBOOK_DIR)
print("Project:", PROJECT_ROOT)
print("Processed Exists:", PROCESSED_DATA.exists())
print("Output Exists:", OUTPUT_DATA.exists())

Notebook: d:\sivahitesh\Desktop\Tail_project\Notebooks
Project: d:\sivahitesh\Desktop\Tail_project
Processed Exists: True
Output Exists: True


In [99]:
# CELL 2: INPUT STRUCTURE VALIDATION
print("=== PHASE 4 INPUT STRUCTURE ===")

print("\nRetail Panel columns:")
print(retail_panel.columns.tolist())

print("\nAudience columns:")
print(audience.columns.tolist())

print("\nDistribution columns:")
print(distribution.columns.tolist())

=== PHASE 4 INPUT STRUCTURE ===

Retail Panel columns:
['audit_timestamp', 'month_id', 'outlet_id', 'numeric_outlet_id', 'district_id', 'district_name', 'state', 'country', 'channel', 'pos_system', 'brand', 'sku_id', 'sku_name', 'pack_size_ml', 'units_sold', 'unit_price_inr', 'mrp_inr', 'discount_pct', 'projection_weight', 'store_audit_notes', 'is_late_submission', 'audit_timestamp_parsed', 'audit_timestamp_is_future', 'district_id_original', 'district_id_normalized', 'district_name_standardized']

Audience columns:
['event_id', 'month_id', 'event_timestamp', 'ingestion_timestamp', 'days_to_ingest', 'district_id', 'district_name', 'state', 'country', 'event_type', 'brand_mentioned', 'user_id', 'user_age', 'device_type', 'screen_resolution', 'session_duration_seconds', 'engagement_clicks', 'campaign_tag', 'feedback_comment', 'event_timestamp_parsed', 'ingestion_timestamp_parsed', 'event_timestamp_is_future', 'ingestion_timestamp_is_future', 'district_id_original', 'district_id_normalize

In [100]:
# === PHASE 4: CATEGORY VELOCITY FIELD CHECK ===

print("=== RETAIL PANEL: VELOCITY-RELATED COLUMNS ===")

velocity_keywords = [
    "category",
    "sku",
    "product",
    "unit",
    "sales",
    "volume",
    "quantity",
    "outlet",
    "month"
]

velocity_columns = [
    col for col in retail_panel.columns
    if any(keyword in col.lower() for keyword in velocity_keywords)
]

for col in velocity_columns:
    print("✓", col)

print("\nTotal candidate columns:", len(velocity_columns))

=== RETAIL PANEL: VELOCITY-RELATED COLUMNS ===
✓ month_id
✓ outlet_id
✓ numeric_outlet_id
✓ sku_id
✓ sku_name
✓ units_sold
✓ unit_price_inr

Total candidate columns: 7


In [101]:
# === PHASE 4: SKU STRUCTURE CHECK ===

sku_summary = (
    retail_panel[["sku_id", "sku_name"]]
    .drop_duplicates()
    .sort_values(["sku_id", "sku_name"])
    .reset_index(drop=True)
)

print("=== UNIQUE SKUs IN RETAIL PANEL ===")
print("Number of unique SKUs:", len(sku_summary))

display(sku_summary)

=== UNIQUE SKUs IN RETAIL PANEL ===
Number of unique SKUs: 29


,sku_id,sku_name
0,APX_BLUE_250,Apex Classic Blue 250ml
1,APX_BLUE_350,Apex Tall 350ml
2,APX_EDITION_RED_250,Apex Watermelon 250ml
3,APX_GOLD_500,Apex Mega Gold 500ml
4,APX_PACK4_250,Apex 4-Can Party Pack
5,APX_SUGARFREE_250,Apex Sugar Free 250ml
6,BLF_EXTRA_350,BullForce Extra Kick 350ml
7,BLF_ORIG_250,BullForce Regular 250ml
8,BLF_ORIG_500,BullForce Big Can 500ml
9,BLF_PUNCH_250,BullForce Fruit Punch 250ml


In [102]:
# === PHASE 4: CATEGORY VELOCITY FIELD VALIDATION ===

required_velocity_fields = [
    "district_id",
    "month_id",
    "sku_id",
    "units_sold",
    "projection_weight"
]

print("=== REQUIRED VELOCITY FIELDS ===")

for col in required_velocity_fields:
    if col in retail_panel.columns:
        print(f"✓ {col}")
    else:
        print(f"✗ {col} NOT FOUND")

print("\n=== PROJECTION WEIGHT SUMMARY ===")

if "projection_weight" in retail_panel.columns:
    print(retail_panel["projection_weight"].describe())
    print("\nMissing projection weights:",
          retail_panel["projection_weight"].isna().sum())

=== REQUIRED VELOCITY FIELDS ===
✓ district_id
✓ month_id
✓ sku_id
✓ units_sold
✓ projection_weight

=== PROJECTION WEIGHT SUMMARY ===
count    1.372163e+06
mean     3.153092e+02
std      7.580920e+02
min      1.200000e-04
25%      1.368600e+02
50%      1.881400e+02
75%      2.784100e+02
max      8.157000e+04
Name: projection_weight, dtype: float64

Missing projection weights: 28130


In [103]:
# === PHASE 4: PROJECTION WEIGHT QUALITY AUDIT ===

weight_missing = retail_panel["projection_weight"].isna()

print("=== PROJECTION WEIGHT QUALITY AUDIT ===")

print("Total Retail Panel rows:", len(retail_panel))
print("Missing projection weights:", weight_missing.sum())
print(
    "Missing weight percentage:",
    round(weight_missing.mean() * 100, 2),
    "%"
)

print("\n=== MISSING WEIGHTS BY DISTRICT ===")

missing_weight_by_district = (
    retail_panel.assign(weight_missing=weight_missing)
    .groupby("district_id", as_index=False)
    .agg(
        total_rows=("weight_missing", "size"),
        missing_weights=("weight_missing", "sum")
    )
)

missing_weight_by_district["missing_weight_pct"] = (
    missing_weight_by_district["missing_weights"]
    / missing_weight_by_district["total_rows"]
    * 100
)

display(
    missing_weight_by_district
    .sort_values("missing_weight_pct", ascending=False)
    .head(20)
)

=== PROJECTION WEIGHT QUALITY AUDIT ===
Total Retail Panel rows: 1400293
Missing projection weights: 28130
Missing weight percentage: 2.01 %

=== MISSING WEIGHTS BY DISTRICT ===


,district_id,total_rows,missing_weights,missing_weight_pct
18,DST_0019,946,33,3.488372
265,DST_0266,1276,38,2.978056
226,DST_0227,620,18,2.903226
318,DST_0319,1703,48,2.818555
117,DST_0118,323,9,2.786378
327,DST_0328,325,9,2.769231
101,DST_0102,701,19,2.710414
158,DST_0159,334,9,2.694611
217,DST_0218,970,26,2.680412
302,DST_0303,301,8,2.657807


In [104]:
# === PHASE 4: CORRECTED MONTHLY CATEGORY VELOCITY ===

# Exclude Kestrel's own SKUs
category_panel = retail_panel[
    ~retail_panel["sku_id"].astype(str).str.startswith("KES_")
].copy()

# Use only rows with valid projection weights
category_panel_weighted = category_panel[
    category_panel["projection_weight"].notna()
].copy()

# Project sampled outlet sales to the estimated panel population
category_panel_weighted["weighted_units_sold"] = (
    category_panel_weighted["units_sold"]
    * category_panel_weighted["projection_weight"]
)

# Step 1: aggregate category volume within each district-month
district_month_velocity = (
    category_panel_weighted
    .groupby(["district_id", "month_id"], as_index=False)
    .agg(
        Weighted_Category_Units=("weighted_units_sold", "sum")
    )
)

# Step 2: calculate average monthly category velocity per district
category_velocity = (
    district_month_velocity
    .groupby("district_id", as_index=False)
    .agg(
        Category_Velocity=("Weighted_Category_Units", "mean"),
        Velocity_Months=("month_id", "nunique")
    )
)

print("=== CORRECTED INDEPENDENT CATEGORY VELOCITY ===")
print("Districts:", category_velocity["district_id"].nunique())
print("District-month records:", len(district_month_velocity))
print(
    "Average months per district:",
    round(category_velocity["Velocity_Months"].mean(), 2)
)

display(
    category_velocity
    .sort_values("Category_Velocity", ascending=False)
    .head(20)
)

=== CORRECTED INDEPENDENT CATEGORY VELOCITY ===
Districts: 340
District-month records: 8160
Average months per district: 24.0


,district_id,Category_Velocity,Velocity_Months
151,DST_0152,2.980824e+07,24
296,DST_0297,1.462685e+07,24
65,DST_0066,1.361672e+07,24
187,DST_0188,1.331584e+07,24
329,DST_0330,1.241004e+07,24
35,DST_0036,1.114447e+07,24
0,DST_0001,1.057407e+07,24
115,DST_0116,1.034134e+07,24
2,DST_0003,8.839759e+06,24
8,DST_0009,8.706947e+06,24


In [10]:
# === PHASE 4: AUDIENCE SIGNAL FIELD CHECK ===

print("=== AUDIENCE SIGNAL: AVAILABLE FIELDS ===")

for col in audience.columns:
    print("✓", col)

print("\n=== AUDIENCE DATA TYPES ===")
print(audience.dtypes)

=== AUDIENCE SIGNAL: AVAILABLE FIELDS ===
✓ event_id
✓ month_id
✓ event_timestamp
✓ ingestion_timestamp
✓ days_to_ingest
✓ district_id
✓ district_name
✓ state
✓ country
✓ event_type
✓ brand_mentioned
✓ user_id
✓ user_age
✓ device_type
✓ screen_resolution
✓ session_duration_seconds
✓ engagement_clicks
✓ campaign_tag
✓ feedback_comment
✓ event_timestamp_parsed
✓ ingestion_timestamp_parsed
✓ event_timestamp_is_future
✓ ingestion_timestamp_is_future
✓ district_id_original
✓ district_id_normalized
✓ district_name_standardized

=== AUDIENCE DATA TYPES ===
event_id                          object
month_id                          object
event_timestamp                   object
ingestion_timestamp               object
days_to_ingest                     int64
district_id                       object
district_name                     object
state                             object
country                           object
event_type                        object
brand_mentioned                   

In [11]:
# === PHASE 4: AUDIENCE EVENT STRUCTURE ===

print("=== EVENT TYPE COUNTS ===")
display(
    audience["event_type"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\n=== BRAND MENTIONED COUNTS ===")
display(
    audience["brand_mentioned"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\n=== ENGAGEMENT CLICKS SUMMARY ===")
print(audience["engagement_clicks"].describe())

print("\n=== DISTRICT-MONTH COVERAGE ===")
print(
    "Unique districts:",
    audience["district_id"].nunique()
)
print(
    "Unique months:",
    audience["month_id"].nunique()
)
print(
    "Unique district-months:",
    audience[["district_id", "month_id"]]
    .drop_duplicates()
    .shape[0]
)

=== EVENT TYPE COUNTS ===


,count
event_type,
campus_event_registration,64638
search_query,64588
influencer_post_engagement,64518
website_session,64401
social_ad_impression,64161
store_locator_lookup,63864



=== BRAND MENTIONED COUNTS ===


,count
brand_mentioned,
Kestrel,77828
Apex Energy,77532
Kestrel Energy,77076
Voltix,76867
General_Energy_Category,76867



=== ENGAGEMENT CLICKS SUMMARY ===
count    3.861700e+05
mean     3.323159e+03
std      1.222041e+05
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      4.500000e+06
Name: engagement_clicks, dtype: float64

=== DISTRICT-MONTH COVERAGE ===
Unique districts: 680
Unique months: 24
Unique district-months: 16240


In [13]:
# === PHASE 4: AUDIENCE EVENT STRUCTURE VALIDATION ===

print("=== AUDIENCE NORMALIZED DISTRICT CHECK ===")

print(
    "Raw district IDs:",
    audience["district_id"].nunique()
)

print(
    "Normalized district IDs:",
    audience["district_id_normalized"].nunique()
)

print(
    "Unique normalized district-months:",
    audience[
        ["district_id_normalized", "month_id"]
    ].drop_duplicates().shape[0]
)

print(
    "Missing normalized district IDs:",
    audience["district_id_normalized"].isna().sum()
)

# Event-level summary
event_by_type = (
    audience
    .groupby("event_type", as_index=False)
    .agg(
        events=("event_id", "count"),
        districts=("district_id_normalized", "nunique")
    )
)

# District-month coverage for each event type
event_month_coverage = (
    audience[
        ["event_type", "district_id_normalized", "month_id"]
    ]
    .drop_duplicates()
    .groupby("event_type")
    .size()
    .reset_index(name="district_months")
)

event_by_type = event_by_type.merge(
    event_month_coverage,
    on="event_type",
    how="left"
)

print("\n=== EVENT TYPES BY NORMALIZED DISTRICT ===")

display(
    event_by_type.sort_values(
        "event_type"
    ).reset_index(drop=True)
)

=== AUDIENCE NORMALIZED DISTRICT CHECK ===
Raw district IDs: 680
Normalized district IDs: 340
Unique normalized district-months: 8160
Missing normalized district IDs: 0

=== EVENT TYPES BY NORMALIZED DISTRICT ===


,event_type,events,districts,district_months
0,campus_event_registration,64638,340,7835
1,influencer_post_engagement,64518,340,7837
2,search_query,64588,340,7866
3,social_ad_impression,64161,340,7841
4,store_locator_lookup,63864,340,7839
5,website_session,64401,340,7874


In [14]:
# === PHASE 4: MONTHLY AUDIENCE SIGNAL ===

# Event types directly aligned with the assessment definition
audience_component_types = [
    "influencer_post_engagement",
    "social_ad_impression",
    "campus_event_registration"
]

audience_components = audience[
    audience["event_type"].isin(audience_component_types)
].copy()

# Aggregate engagement activity by district and month
audience_monthly = (
    audience_components
    .groupby(
        ["district_id_normalized", "month_id", "event_type"],
        as_index=False
    )
    .agg(
        Event_Count=("event_id", "count"),
        Total_Engagement_Clicks=("engagement_clicks", "sum")
    )
)

print("=== MONTHLY AUDIENCE COMPONENTS ===")
print("Rows:", len(audience_monthly))
print(
    "Districts:",
    audience_monthly["district_id_normalized"].nunique()
)
print(
    "District-months:",
    audience_monthly[
        ["district_id_normalized", "month_id"]
    ].drop_duplicates().shape[0]
)

display(
    audience_monthly.head(20)
)

=== MONTHLY AUDIENCE COMPONENTS ===
Rows: 23513
Districts: 340
District-months: 8135


,district_id_normalized,month_id,event_type,Event_Count,Total_Engagement_Clicks
0,DST_0001,2024-09,campus_event_registration,14,14
1,DST_0001,2024-09,influencer_post_engagement,10,10
2,DST_0001,2024-09,social_ad_impression,17,17
3,DST_0001,2024-10,campus_event_registration,14,14
4,DST_0001,2024-10,influencer_post_engagement,25,34
5,DST_0001,2024-10,social_ad_impression,12,12
6,DST_0001,2024-11,campus_event_registration,13,13
7,DST_0001,2024-11,influencer_post_engagement,17,66
8,DST_0001,2024-11,social_ad_impression,17,66
9,DST_0001,2024-12,campus_event_registration,15,64


In [16]:
# === PHASE 4: COMPLETE AUDIENCE DISTRICT-MONTH GRID ===

# Load processed District Master if not already available
district_master = pd.read_csv(
    PROCESSED_DATA / "district_master_processed.csv"
)

# Create the canonical 340-district × 24-month grid
district_ids = (
    district_master["District_Code"]
    .astype(str)
    .unique()
)

months = sorted(
    audience["month_id"]
    .dropna()
    .unique()
)

audience_grid = pd.MultiIndex.from_product(
    [district_ids, months],
    names=["district_id_normalized", "month_id"]
).to_frame(index=False)

# Pivot the three required audience components
audience_pivot = (
    audience_monthly
    .pivot_table(
        index=["district_id_normalized", "month_id"],
        columns="event_type",
        values=["Event_Count", "Total_Engagement_Clicks"],
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

# Flatten column names
audience_pivot.columns = [
    "_".join(col).strip("_")
    if isinstance(col, tuple)
    else col
    for col in audience_pivot.columns
]

# Merge with complete 340 × 24 grid
audience_monthly_complete = audience_grid.merge(
    audience_pivot,
    on=["district_id_normalized", "month_id"],
    how="left"
)

# Missing event activity = zero observed activity
numeric_columns = audience_monthly_complete.select_dtypes(
    include="number"
).columns

audience_monthly_complete[numeric_columns] = (
    audience_monthly_complete[numeric_columns]
    .fillna(0)
)

print("=== COMPLETE AUDIENCE DISTRICT-MONTH GRID ===")
print(
    "Districts:",
    audience_monthly_complete["district_id_normalized"].nunique()
)
print(
    "Months:",
    audience_monthly_complete["month_id"].nunique()
)
print(
    "District-months:",
    len(audience_monthly_complete)
)

print("\nMissing numeric values:")
print(
    audience_monthly_complete[numeric_columns]
    .isna()
    .sum()
)

display(audience_monthly_complete.head(20))

=== COMPLETE AUDIENCE DISTRICT-MONTH GRID ===
Districts: 340
Months: 24
District-months: 8160

Missing numeric values:
Event_Count_campus_event_registration                 0
Event_Count_influencer_post_engagement                0
Event_Count_social_ad_impression                      0
Total_Engagement_Clicks_campus_event_registration     0
Total_Engagement_Clicks_influencer_post_engagement    0
Total_Engagement_Clicks_social_ad_impression          0
dtype: int64


,district_id_normalized,month_id,Event_Count_campus_event_registration,Event_Count_influencer_post_engagement,Event_Count_social_ad_impression,Total_Engagement_Clicks_campus_event_registration,Total_Engagement_Clicks_influencer_post_engagement,Total_Engagement_Clicks_social_ad_impression
0,DST_0001,2024-09,14.0,10.0,17.0,14.0,10.0,17.0
1,DST_0001,2024-10,14.0,25.0,12.0,14.0,34.0,12.0
2,DST_0001,2024-11,13.0,17.0,17.0,13.0,66.0,66.0
3,DST_0001,2024-12,15.0,11.0,16.0,64.0,11.0,16.0
4,DST_0001,2025-01,13.0,14.0,14.0,13.0,14.0,14.0
5,DST_0001,2025-02,12.0,21.0,17.0,12.0,39.0,17.0
6,DST_0001,2025-03,17.0,15.0,13.0,17.0,33.0,13.0
7,DST_0001,2025-04,15.0,15.0,13.0,15.0,113.0,62.0
8,DST_0001,2025-05,10.0,18.0,18.0,10.0,27.0,18.0
9,DST_0001,2025-06,16.0,17.0,10.0,25.0,26.0,10.0


In [17]:
# === PHASE 4: DISTRICT-LEVEL AUDIENCE COMPONENTS ===

audience_components_24m = (
    audience_monthly_complete
    .groupby("district_id_normalized", as_index=False)
    .agg(
        Campus_Event_Registrations=(
            "Event_Count_campus_event_registration",
            "sum"
        ),
        Influencer_Engagement=(
            "Total_Engagement_Clicks_influencer_post_engagement",
            "sum"
        ),
        Social_Ad_Impressions=(
            "Event_Count_social_ad_impression",
            "sum"
        )
    )
)

print("=== 24-MONTH AUDIENCE COMPONENTS ===")

print(
    "Districts:",
    audience_components_24m["district_id_normalized"].nunique()
)

print("\nSummary statistics:")
display(
    audience_components_24m[
        [
            "Campus_Event_Registrations",
            "Influencer_Engagement",
            "Social_Ad_Impressions"
        ]
    ].describe()
)

display(audience_components_24m.head(20))

=== 24-MONTH AUDIENCE COMPONENTS ===
Districts: 340

Summary statistics:


,Campus_Event_Registrations,Influencer_Engagement,Social_Ad_Impressions
count,340.000000,3.400000e+02,340.000000
mean,190.111765,6.224511e+05,188.708824
std,108.449705,1.837186e+06,106.984135
min,10.000000,1.100000e+01,12.000000
25%,107.000000,2.130000e+02,100.000000
50%,167.000000,3.935000e+02,169.500000
75%,271.250000,6.565000e+02,267.250000
max,437.000000,1.350039e+07,445.000000


,district_id_normalized,Campus_Event_Registrations,Influencer_Engagement,Social_Ad_Impressions
0,DST_0001,366.0,718.0,382.0
1,DST_0002,405.0,4500925.0,339.0
2,DST_0003,422.0,4500715.0,370.0
3,DST_0004,407.0,649.0,397.0
4,DST_0005,332.0,744.0,373.0
5,DST_0006,294.0,428.0,345.0
6,DST_0007,368.0,815.0,421.0
7,DST_0008,401.0,968.0,388.0
8,DST_0009,366.0,545.0,390.0
9,DST_0010,321.0,526.0,312.0


In [18]:
# === PHASE 4: AUDIENCE OUTLIER AUDIT ===

print("=== AUDIENCE COMPONENT OUTLIER AUDIT ===")

for col in [
    "Campus_Event_Registrations",
    "Influencer_Engagement",
    "Social_Ad_Impressions"
]:
    q1 = audience_components_24m[col].quantile(0.25)
    q3 = audience_components_24m[col].quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + (1.5 * iqr)

    outliers = (
        audience_components_24m[col] > upper_bound
    ).sum()

    print(f"\n{col}")
    print("Q1:", q1)
    print("Q3:", q3)
    print("IQR:", iqr)
    print("Upper IQR bound:", upper_bound)
    print("Outlier districts:", outliers)
    print("Maximum:", audience_components_24m[col].max())

print("\n=== TOP INFLUENCER ENGAGEMENT DISTRICTS ===")

display(
    audience_components_24m[
        [
            "district_id_normalized",
            "Influencer_Engagement"
        ]
    ]
    .sort_values(
        "Influencer_Engagement",
        ascending=False
    )
    .head(20)
)

=== AUDIENCE COMPONENT OUTLIER AUDIT ===

Campus_Event_Registrations
Q1: 107.0
Q3: 271.25
IQR: 164.25
Upper IQR bound: 517.625
Outlier districts: 0
Maximum: 437.0

Influencer_Engagement
Q1: 213.0
Q3: 656.5
IQR: 443.5
Upper IQR bound: 1321.75
Outlier districts: 40
Maximum: 13500391.0

Social_Ad_Impressions
Q1: 100.0
Q3: 267.25
IQR: 167.25
Upper IQR bound: 518.125
Outlier districts: 0
Maximum: 445.0

=== TOP INFLUENCER ENGAGEMENT DISTRICTS ===


,district_id_normalized,Influencer_Engagement
166,DST_0167,13500391.0
193,DST_0194,9000827.0
76,DST_0077,9000742.0
137,DST_0138,9000471.0
199,DST_0200,9000308.0
211,DST_0212,9000176.0
191,DST_0192,4501026.0
1,DST_0002,4500925.0
114,DST_0115,4500766.0
150,DST_0151,4500763.0


In [19]:
# === PHASE 4: AUDIENCE SIGNAL vs MARKET SIZE ===

audience_market_check = audience_components_24m.merge(
    district_master[
        [
            "District_Code",
            "Total_Population",
            "Estimated_Total_Retail_Universe"
        ]
    ],
    left_on="district_id_normalized",
    right_on="District_Code",
    how="left",
    validate="one_to_one"
)

# Per-population audience measures
audience_market_check["Influencer_per_100k"] = (
    audience_market_check["Influencer_Engagement"]
    / audience_market_check["Total_Population"]
    * 100000
)

audience_market_check["Events_per_100k"] = (
    audience_market_check["Campus_Event_Registrations"]
    / audience_market_check["Total_Population"]
    * 100000
)

audience_market_check["Social_Impressions_per_100k"] = (
    audience_market_check["Social_Ad_Impressions"]
    / audience_market_check["Total_Population"]
    * 100000
)

print("=== AUDIENCE SIGNAL vs MARKET SIZE ===")

print(
    "Missing population:",
    audience_market_check["Total_Population"].isna().sum()
)

print(
    "Missing retail universe:",
    audience_market_check["Estimated_Total_Retail_Universe"].isna().sum()
)

print("\n=== TOP INFLUENCER ENGAGEMENT ===")

display(
    audience_market_check[
        [
            "district_id_normalized",
            "Total_Population",
            "Influencer_Engagement",
            "Influencer_per_100k"
        ]
    ]
    .sort_values(
        "Influencer_Engagement",
        ascending=False
    )
    .head(20)
)

=== AUDIENCE SIGNAL vs MARKET SIZE ===
Missing population: 0
Missing retail universe: 0

=== TOP INFLUENCER ENGAGEMENT ===


,district_id_normalized,Total_Population,Influencer_Engagement,Influencer_per_100k
166,DST_0167,3120506,13500391.0,432634.675274
193,DST_0194,7103808,9000827.0,126704.254957
76,DST_0077,2556244,9000742.0,352108.092968
137,DST_0138,3405559,9000471.0,264287.625027
199,DST_0200,2819086,9000308.0,319263.335705
211,DST_0212,1081852,9000176.0,831923.035683
191,DST_0192,2882031,4501026.0,156175.488744
1,DST_0002,3085411,4500925.0,145877.648067
114,DST_0115,4418797,4500766.0,101855.007143
150,DST_0151,7214225,4500763.0,62387.338903


In [20]:
# === PHASE 4: MARKET-SIZE NORMALIZED AUDIENCE COMPONENTS ===

# Normalize audience activity by district population
audience_market_check["Influencer_per_100k"] = (
    audience_market_check["Influencer_Engagement"]
    / audience_market_check["Total_Population"]
    * 100000
)

audience_market_check["Events_per_100k"] = (
    audience_market_check["Campus_Event_Registrations"]
    / audience_market_check["Total_Population"]
    * 100000
)

audience_market_check["Social_Impressions_per_100k"] = (
    audience_market_check["Social_Ad_Impressions"]
    / audience_market_check["Total_Population"]
    * 100000
)

# Robust normalization using percentile rank
audience_market_check["Influencer_Score"] = (
    audience_market_check["Influencer_per_100k"]
    .rank(pct=True, method="average")
)

audience_market_check["Event_Score"] = (
    audience_market_check["Events_per_100k"]
    .rank(pct=True, method="average")
)

audience_market_check["Social_Score"] = (
    audience_market_check["Social_Impressions_per_100k"]
    .rank(pct=True, method="average")
)

# Equal-weight audience signal
audience_market_check["Audience_Signal"] = (
    audience_market_check[
        [
            "Influencer_Score",
            "Event_Score",
            "Social_Score"
        ]
    ].mean(axis=1)
)

print("=== NORMALIZED AUDIENCE SIGNAL ===")

print(
    "Districts:",
    audience_market_check["district_id_normalized"].nunique()
)

print("\nScore ranges:")

for col in [
    "Influencer_Score",
    "Event_Score",
    "Social_Score",
    "Audience_Signal"
]:
    print(
        f"{col}:",
        round(audience_market_check[col].min(), 4),
        "to",
        round(audience_market_check[col].max(), 4)
    )

print("\n=== TOP AUDIENCE SIGNAL DISTRICTS ===")

display(
    audience_market_check[
        [
            "district_id_normalized",
            "Influencer_per_100k",
            "Events_per_100k",
            "Social_Impressions_per_100k",
            "Audience_Signal"
        ]
    ]
    .sort_values(
        "Audience_Signal",
        ascending=False
    )
    .head(20)
)

=== NORMALIZED AUDIENCE SIGNAL ===
Districts: 340

Score ranges:
Influencer_Score: 0.0029 to 1.0
Event_Score: 0.0029 to 1.0
Social_Score: 0.0029 to 1.0
Audience_Signal: 0.0098 to 0.9618

=== TOP AUDIENCE SIGNAL DISTRICTS ===


,district_id_normalized,Influencer_per_100k,Events_per_100k,Social_Impressions_per_100k,Audience_Signal
75,DST_0076,385827.772781,14.231812,13.631676,0.961765
99,DST_0100,77.792279,30.910839,27.819755,0.960784
210,DST_0211,44.547274,25.614683,24.501001,0.956863
39,DST_0040,37.303271,19.003553,18.299718,0.947059
40,DST_0041,215370.954644,13.925784,13.782219,0.947059
212,DST_0213,39.872289,17.102329,18.295514,0.946078
272,DST_0273,41.702535,16.307987,17.503587,0.945098
302,DST_0303,47.834804,15.284115,14.900413,0.942157
240,DST_0241,36.237567,17.478091,16.504238,0.940196
191,DST_0192,156175.488744,15.162918,12.491191,0.939216


In [21]:
# === PHASE 4: CHECK EXTREME AUDIENCE ENGAGEMENT IN PROCESSED DATA ===

print("=== PROCESSED AUDIENCE ENGAGEMENT QUALITY CHECK ===")

engagement_median = audience["engagement_clicks"].median()
engagement_threshold = engagement_median * 100

extreme_rows = audience[
    audience["engagement_clicks"] > engagement_threshold
].copy()

print("Median engagement clicks:", engagement_median)
print("100× median threshold:", engagement_threshold)
print("Rows above 100× median:", len(extreme_rows))
print(
    "Percentage of processed audience:",
    round(len(extreme_rows) / len(audience) * 100, 4),
    "%"
)

print("\nTop extreme engagement rows:")

display(
    extreme_rows[
        [
            "event_id",
            "district_id_normalized",
            "event_type",
            "engagement_clicks"
        ]
    ]
    .sort_values("engagement_clicks", ascending=False)
    .head(20)
)

=== PROCESSED AUDIENCE ENGAGEMENT QUALITY CHECK ===
Median engagement clicks: 1.0
100× median threshold: 100.0
Rows above 100× median: 285
Percentage of processed audience: 0.0738 %

Top extreme engagement rows:


,event_id,district_id_normalized,event_type,engagement_clicks
269,EVT_00031481,DST_0312,influencer_post_engagement,4500000
877,EVT_00179641,DST_0049,store_locator_lookup,4500000
6701,EVT_00075515,DST_0194,social_ad_impression,4500000
6702,EVT_00318301,DST_0327,website_session,4500000
9067,EVT_00084603,DST_0063,campus_event_registration,4500000
9140,EVT_00100989,DST_0066,website_session,4500000
10439,EVT_00234129,DST_0190,campus_event_registration,4500000
10779,EVT_00304017,DST_0008,search_query,4500000
11012,EVT_00121583,DST_0149,search_query,4500000
12595,EVT_00066862,DST_0020,store_locator_lookup,4500000


In [22]:
# === PHASE 4: EXTREME ENGAGEMENT BY EVENT TYPE ===

extreme_engagement = audience[
    audience["engagement_clicks"] > engagement_threshold
].copy()

extreme_by_type = (
    extreme_engagement
    .groupby("event_type", as_index=False)
    .agg(
        Extreme_Rows=("event_id", "count"),
        Max_Engagement=("engagement_clicks", "max"),
        Median_Extreme_Engagement=("engagement_clicks", "median")
    )
    .sort_values("Extreme_Rows", ascending=False)
)

print("=== EXTREME ENGAGEMENT BY EVENT TYPE ===")
print("Total extreme rows:", len(extreme_engagement))

display(extreme_by_type)

print("\n=== EXTREME ENGAGEMENT VALUE FREQUENCY ===")

display(
    extreme_engagement["engagement_clicks"]
    .value_counts()
    .head(20)
    .to_frame("rows")
)

=== EXTREME ENGAGEMENT BY EVENT TYPE ===
Total extreme rows: 285


,event_type,Extreme_Rows,Max_Engagement,Median_Extreme_Engagement
4,store_locator_lookup,56,4500000,4500000.0
0,campus_event_registration,53,4500000,4500000.0
1,influencer_post_engagement,47,4500000,4500000.0
3,social_ad_impression,44,4500000,4500000.0
5,website_session,43,4500000,4500000.0
2,search_query,42,4500000,4500000.0



=== EXTREME ENGAGEMENT VALUE FREQUENCY ===


,rows
engagement_clicks,
4500000,285


In [23]:
# === PHASE 4: CLEAN AUDIENCE ENGAGEMENT FOR ANALYSIS ===

audience_analysis = audience.copy()

# Identify impossible engagement values
engagement_median = audience_analysis["engagement_clicks"].median()
engagement_threshold = engagement_median * 100

extreme_mask = (
    audience_analysis["engagement_clicks"]
    > engagement_threshold
)

extreme_count = extreme_mask.sum()

# Keep the event, but remove the impossible engagement contribution
audience_analysis.loc[
    extreme_mask,
    "engagement_clicks"
] = 0

print("=== AUDIENCE ANALYSIS CLEANING ===")
print("Original audience rows:", len(audience))
print("Extreme engagement rows identified:", extreme_count)
print("Engagement threshold:", engagement_threshold)
print(
    "Extreme values remaining:",
    (
        audience_analysis["engagement_clicks"]
        > engagement_threshold
    ).sum()
)

print("\nOriginal processed dataset unchanged:", len(audience) == len(audience_analysis))

=== AUDIENCE ANALYSIS CLEANING ===
Original audience rows: 386170
Extreme engagement rows identified: 285
Engagement threshold: 100.0
Extreme values remaining: 0

Original processed dataset unchanged: True


In [24]:
# === PHASE 4: REBUILD CLEAN MONTHLY AUDIENCE COMPONENTS ===

audience_components_clean = audience_analysis[
    audience_analysis["event_type"].isin(
        [
            "influencer_post_engagement",
            "social_ad_impression",
            "campus_event_registration"
        ]
    )
].copy()

audience_monthly_clean = (
    audience_components_clean
    .groupby(
        [
            "district_id_normalized",
            "month_id",
            "event_type"
        ],
        as_index=False
    )
    .agg(
        Event_Count=("event_id", "count"),
        Total_Engagement_Clicks=("engagement_clicks", "sum")
    )
)

print("=== CLEAN MONTHLY AUDIENCE COMPONENTS ===")
print("Rows:", len(audience_monthly_clean))
print(
    "Districts:",
    audience_monthly_clean["district_id_normalized"].nunique()
)
print(
    "District-months:",
    audience_monthly_clean[
        ["district_id_normalized", "month_id"]
    ].drop_duplicates().shape[0]
)

display(
    audience_monthly_clean.head(20)
)

=== CLEAN MONTHLY AUDIENCE COMPONENTS ===
Rows: 23513
Districts: 340
District-months: 8135


,district_id_normalized,month_id,event_type,Event_Count,Total_Engagement_Clicks
0,DST_0001,2024-09,campus_event_registration,14,14
1,DST_0001,2024-09,influencer_post_engagement,10,10
2,DST_0001,2024-09,social_ad_impression,17,17
3,DST_0001,2024-10,campus_event_registration,14,14
4,DST_0001,2024-10,influencer_post_engagement,25,34
5,DST_0001,2024-10,social_ad_impression,12,12
6,DST_0001,2024-11,campus_event_registration,13,13
7,DST_0001,2024-11,influencer_post_engagement,17,66
8,DST_0001,2024-11,social_ad_impression,17,66
9,DST_0001,2024-12,campus_event_registration,15,64


In [25]:
# === PHASE 4: COMPLETE CLEAN AUDIENCE GRID ===

# Create the canonical 340 × 24 district-month grid
district_ids = (
    district_master["District_Code"]
    .astype(str)
    .unique()
)

months = sorted(
    audience_analysis["month_id"]
    .dropna()
    .unique()
)

audience_grid_clean = pd.MultiIndex.from_product(
    [district_ids, months],
    names=["district_id_normalized", "month_id"]
).to_frame(index=False)

# Pivot cleaned monthly audience components
audience_pivot_clean = (
    audience_monthly_clean
    .pivot_table(
        index=[
            "district_id_normalized",
            "month_id"
        ],
        columns="event_type",
        values=[
            "Event_Count",
            "Total_Engagement_Clicks"
        ],
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

# Flatten column names
audience_pivot_clean.columns = [
    "_".join(col).strip("_")
    if isinstance(col, tuple)
    else col
    for col in audience_pivot_clean.columns
]

# Merge onto complete grid
audience_monthly_complete_clean = audience_grid_clean.merge(
    audience_pivot_clean,
    on=[
        "district_id_normalized",
        "month_id"
    ],
    how="left"
)

# No recorded activity = zero observed activity
numeric_columns = audience_monthly_complete_clean.select_dtypes(
    include="number"
).columns

audience_monthly_complete_clean[numeric_columns] = (
    audience_monthly_complete_clean[numeric_columns]
    .fillna(0)
)

print("=== COMPLETE CLEAN AUDIENCE GRID ===")
print(
    "Districts:",
    audience_monthly_complete_clean[
        "district_id_normalized"
    ].nunique()
)
print(
    "Months:",
    audience_monthly_complete_clean[
        "month_id"
    ].nunique()
)
print(
    "District-months:",
    len(audience_monthly_complete_clean)
)

print("\nMissing numeric values:")
print(
    audience_monthly_complete_clean[
        numeric_columns
    ].isna().sum()
)

display(
    audience_monthly_complete_clean.head(20)
)

=== COMPLETE CLEAN AUDIENCE GRID ===
Districts: 340
Months: 24
District-months: 8160

Missing numeric values:
Event_Count_campus_event_registration                 0
Event_Count_influencer_post_engagement                0
Event_Count_social_ad_impression                      0
Total_Engagement_Clicks_campus_event_registration     0
Total_Engagement_Clicks_influencer_post_engagement    0
Total_Engagement_Clicks_social_ad_impression          0
dtype: int64


,district_id_normalized,month_id,Event_Count_campus_event_registration,Event_Count_influencer_post_engagement,Event_Count_social_ad_impression,Total_Engagement_Clicks_campus_event_registration,Total_Engagement_Clicks_influencer_post_engagement,Total_Engagement_Clicks_social_ad_impression
0,DST_0001,2024-09,14.0,10.0,17.0,14.0,10.0,17.0
1,DST_0001,2024-10,14.0,25.0,12.0,14.0,34.0,12.0
2,DST_0001,2024-11,13.0,17.0,17.0,13.0,66.0,66.0
3,DST_0001,2024-12,15.0,11.0,16.0,64.0,11.0,16.0
4,DST_0001,2025-01,13.0,14.0,14.0,13.0,14.0,14.0
5,DST_0001,2025-02,12.0,21.0,17.0,12.0,39.0,17.0
6,DST_0001,2025-03,17.0,15.0,13.0,17.0,33.0,13.0
7,DST_0001,2025-04,15.0,15.0,13.0,15.0,113.0,62.0
8,DST_0001,2025-05,10.0,18.0,18.0,10.0,27.0,18.0
9,DST_0001,2025-06,16.0,17.0,10.0,25.0,26.0,10.0


In [26]:
# === PHASE 4: MARKET-SIZE NORMALIZED CATEGORY VELOCITY ===

# Add market-size information from District Master
category_velocity_market = category_velocity.merge(
    district_master[
        [
            "District_Code",
            "Estimated_Total_Retail_Universe"
        ]
    ],
    left_on="district_id",
    right_on="District_Code",
    how="left",
    validate="one_to_one"
)

# Validate the join
print("=== CATEGORY VELOCITY MARKET-SIZE JOIN ===")
print(
    "Districts:",
    category_velocity_market["district_id"].nunique()
)
print(
    "Missing retail universe:",
    category_velocity_market[
        "Estimated_Total_Retail_Universe"
    ].isna().sum()
)

# Normalize category velocity by estimated retail universe
category_velocity_market["Velocity_per_Retail_Outlet"] = (
    category_velocity_market["Category_Velocity"]
    / category_velocity_market["Estimated_Total_Retail_Universe"]
)

# Convert to percentile score for comparability
category_velocity_market["Category_Velocity_Score"] = (
    category_velocity_market["Velocity_per_Retail_Outlet"]
    .rank(pct=True, method="average")
)

print("\n=== NORMALIZED CATEGORY VELOCITY ===")

print(
    "Score range:",
    round(
        category_velocity_market["Category_Velocity_Score"].min(),
        4
    ),
    "to",
    round(
        category_velocity_market["Category_Velocity_Score"].max(),
        4
    )
)

display(
    category_velocity_market[
        [
            "district_id",
            "Category_Velocity",
            "Estimated_Total_Retail_Universe",
            "Velocity_per_Retail_Outlet",
            "Category_Velocity_Score"
        ]
    ]
    .sort_values(
        "Category_Velocity_Score",
        ascending=False
    )
    .head(20)
)

=== CATEGORY VELOCITY MARKET-SIZE JOIN ===
Districts: 340
Missing retail universe: 0

=== NORMALIZED CATEGORY VELOCITY ===
Score range: 0.0029 to 1.0


,district_id,Category_Velocity,Estimated_Total_Retail_Universe,Velocity_per_Retail_Outlet,Category_Velocity_Score
151,DST_0152,2.980824e+07,11676,2552.949423,1.000000
240,DST_0241,8.143477e+06,3199,2545.632004,0.997059
296,DST_0297,1.462685e+07,6016,2431.324711,0.994118
329,DST_0330,1.241004e+07,5424,2287.987443,0.991176
65,DST_0066,1.361672e+07,6445,2112.757822,0.988235
115,DST_0116,1.034134e+07,5213,1983.759380,0.985294
271,DST_0272,7.796935e+06,3948,1974.907588,0.982353
242,DST_0243,6.594644e+06,3512,1877.745971,0.979412
298,DST_0299,7.160671e+06,3970,1803.695487,0.976471
8,DST_0009,8.706947e+06,4837,1800.071682,0.973529


In [27]:
# === PHASE 4: INDEPENDENT DEMAND SIGNAL ===

# Combine the two independent demand pillars:
# 1. Category velocity
# 2. Audience engagement + event registrations

demand_signal = category_velocity_market[
    [
        "district_id",
        "Category_Velocity",
        "Estimated_Total_Retail_Universe",
        "Velocity_per_Retail_Outlet",
        "Category_Velocity_Score"
    ]
].merge(
    audience_market_check[
        [
            "district_id_normalized",
            "Audience_Signal"
        ]
    ],
    left_on="district_id",
    right_on="district_id_normalized",
    how="left",
    validate="one_to_one"
)

# Equal-weight combination
demand_signal["Demand_Signal"] = (
    0.50 * demand_signal["Category_Velocity_Score"]
    + 0.50 * demand_signal["Audience_Signal"]
)

print("=== INDEPENDENT DEMAND SIGNAL ===")

print(
    "Districts:",
    demand_signal["district_id"].nunique()
)

print(
    "Missing Category Velocity Score:",
    demand_signal["Category_Velocity_Score"].isna().sum()
)

print(
    "Missing Audience Signal:",
    demand_signal["Audience_Signal"].isna().sum()
)

print(
    "Demand Signal range:",
    round(demand_signal["Demand_Signal"].min(), 4),
    "to",
    round(demand_signal["Demand_Signal"].max(), 4)
)

print("\n=== TOP DEMAND DISTRICTS ===")

display(
    demand_signal[
        [
            "district_id",
            "Category_Velocity_Score",
            "Audience_Signal",
            "Demand_Signal"
        ]
    ]
    .sort_values(
        "Demand_Signal",
        ascending=False
    )
    .head(20)
)

=== INDEPENDENT DEMAND SIGNAL ===
Districts: 340
Missing Category Velocity Score: 0
Missing Audience Signal: 0
Demand Signal range: 0.0088 to 0.9686

=== TOP DEMAND DISTRICTS ===


,district_id,Category_Velocity_Score,Audience_Signal,Demand_Signal
240,DST_0241,0.997059,0.940196,0.968627
271,DST_0272,0.982353,0.934314,0.958333
99,DST_0100,0.952941,0.960784,0.956863
272,DST_0273,0.964706,0.945098,0.954902
212,DST_0213,0.958824,0.946078,0.952451
191,DST_0192,0.950000,0.939216,0.944608
39,DST_0040,0.923529,0.947059,0.935294
242,DST_0243,0.979412,0.887255,0.933333
329,DST_0330,0.991176,0.869608,0.930392
296,DST_0297,0.994118,0.865686,0.929902


In [28]:
# === PHASE 4: DISTRIBUTION GAP INPUT CHECK ===

print("=== DISTRIBUTION DATA STRUCTURE ===")

print("Rows:", len(distribution))
print("Columns:")
print(distribution.columns.tolist())

print("\n=== DATA TYPES ===")
print(distribution.dtypes)

print("\n=== KEY DISTRIBUTION FIELDS ===")

candidate_columns = [
    "District_ID",
    "District_Name",
    "Outlets_Stocking_Kestrel",
    "Outlets_Stocking_Category",
    "Estimated_Total_Universe"
]

for col in candidate_columns:
    if col in distribution.columns:
        print("✓", col)

print("\n=== UNIQUE AUDIT CYCLES ===")

if "Audit_Cycle" in distribution.columns:
    print(distribution["Audit_Cycle"].value_counts(dropna=False))

print("\n=== SAMPLE DISTRIBUTION RECORDS ===")
display(distribution.head(20))

=== DISTRIBUTION DATA STRUCTURE ===
Rows: 8160
Columns:
['Audit_Cycle', 'District_ID', 'District_Name', 'State', 'Outlets_Stocking_Kestrel', 'Outlets_Stocking_Category', 'Estimated_Total_Universe', 'Apparent_Kestrel_Numeric_Dist_Pct', 'Apparent_Category_Numeric_Dist_Pct', 'distribution_data_quality_issue']

=== DATA TYPES ===
Audit_Cycle                            object
District_ID                            object
District_Name                          object
State                                  object
Outlets_Stocking_Kestrel                int64
Outlets_Stocking_Category               int64
Estimated_Total_Universe                int64
Apparent_Kestrel_Numeric_Dist_Pct     float64
Apparent_Category_Numeric_Dist_Pct    float64
distribution_data_quality_issue          bool
dtype: object

=== KEY DISTRIBUTION FIELDS ===
✓ District_ID
✓ District_Name
✓ Outlets_Stocking_Kestrel
✓ Outlets_Stocking_Category
✓ Estimated_Total_Universe

=== UNIQUE AUDIT CYCLES ===
Audit_Cycle
2024-09    3

,Audit_Cycle,District_ID,District_Name,State,Outlets_Stocking_Kestrel,Outlets_Stocking_Category,Estimated_Total_Universe,Apparent_Kestrel_Numeric_Dist_Pct,Apparent_Category_Numeric_Dist_Pct,distribution_data_quality_issue
0,2024-09,DST_0001,Mumbai Suburban,Maharashtra,9976,15117,18500,53.92,81.71,False
1,2024-09,DST_0002,Mumbai City,Maharashtra,3629,5346,6787,53.47,78.77,False
2,2024-09,DST_0003,Pune,Maharashtra,7037,12407,15596,45.12,79.55,False
3,2024-09,DST_0004,Thane,Maharashtra,4306,10835,15155,28.41,71.49,False
4,2024-09,DST_0005,Palghar,Maharashtra,1231,3545,4401,27.97,80.55,False
5,2024-09,DST_0006,Nagpur,Maharashtra,1725,5465,8153,21.16,67.03,False
6,2024-09,DST_0007,Nashik,Maharashtra,425,6475,8562,4.96,75.62,False
7,2024-09,DST_0008,Aurangabad,Maharashtra,409,3705,5241,7.80,70.69,False
8,2024-09,DST_0009,Kolhapur,Maharashtra,1474,4272,4837,30.47,88.32,False
9,2024-09,DST_0010,Solapur,Maharashtra,1662,3442,5388,30.85,63.88,False


In [29]:
# === PHASE 4: DISTRIBUTION QUALITY AUDIT ===

print("=== DISTRIBUTION QUALITY AUDIT ===")

print("Total rows:", len(distribution))

print(
    "Unique districts:",
    distribution["District_ID"].nunique()
)

print(
    "Unique audit cycles:",
    distribution["Audit_Cycle"].nunique()
)

print(
    "Unique District × Audit Cycle:",
    distribution[
        ["District_ID", "Audit_Cycle"]
    ].drop_duplicates().shape[0]
)

print("\n=== DISTRIBUTION DATA-QUALITY FLAGS ===")

print(
    "Rows flagged with distribution_data_quality_issue:",
    distribution["distribution_data_quality_issue"].sum()
)

print(
    "Percentage flagged:",
    round(
        distribution["distribution_data_quality_issue"].mean() * 100,
        2
    ),
    "%"
)

print("\n=== LOGICAL DISTRIBUTION CHECKS ===")

# Kestrel stocking cannot exceed category stocking
kestrel_gt_category = (
    distribution["Outlets_Stocking_Kestrel"]
    > distribution["Outlets_Stocking_Category"]
)

# Category stocking cannot exceed estimated universe
category_gt_universe = (
    distribution["Outlets_Stocking_Category"]
    > distribution["Estimated_Total_Universe"]
)

# Kestrel stocking cannot exceed estimated universe
kestrel_gt_universe = (
    distribution["Outlets_Stocking_Kestrel"]
    > distribution["Estimated_Total_Universe"]
)

print(
    "Kestrel outlets > category outlets:",
    kestrel_gt_category.sum()
)

print(
    "Category outlets > estimated universe:",
    category_gt_universe.sum()
)

print(
    "Kestrel outlets > estimated universe:",
    kestrel_gt_universe.sum()
)

print("\n=== FLAGGED DISTRIBUTION RECORDS ===")

display(
    distribution[
        distribution["distribution_data_quality_issue"]
    ].head(20)
)

=== DISTRIBUTION QUALITY AUDIT ===
Total rows: 8160
Unique districts: 340
Unique audit cycles: 24
Unique District × Audit Cycle: 8160

=== DISTRIBUTION DATA-QUALITY FLAGS ===
Rows flagged with distribution_data_quality_issue: 159
Percentage flagged: 1.95 %

=== LOGICAL DISTRIBUTION CHECKS ===
Kestrel outlets > category outlets: 122
Category outlets > estimated universe: 37
Kestrel outlets > estimated universe: 19

=== FLAGGED DISTRIBUTION RECORDS ===


,Audit_Cycle,District_ID,District_Name,State,Outlets_Stocking_Kestrel,Outlets_Stocking_Category,Estimated_Total_Universe,Apparent_Kestrel_Numeric_Dist_Pct,Apparent_Category_Numeric_Dist_Pct,distribution_data_quality_issue
16,2024-09,DST_0017,Raigad,Maharashtra,757,4077,3471,21.81,117.46,True
21,2024-09,DST_0022,Buldhana,Maharashtra,1822,1784,2829,64.40,63.06,True
61,2024-09,DST_0062,Chamarajanagar,Karnataka,671,553,1059,63.36,52.22,True
87,2024-09,DST_0088,Ranipet,Tamil Nadu,486,2426,1679,28.95,144.49,True
204,2024-09,DST_0205,Purulia,West Bengal,1707,1479,2877,59.33,51.41,True
361,2024-10,DST_0022,Buldhana,Maharashtra,3171,1923,2829,112.09,67.97,True
371,2024-10,DST_0032,Nandurbar,Maharashtra,1820,910,1710,106.43,53.22,True
468,2024-10,DST_0129,Budaun,Uttar Pradesh,2179,2089,3821,57.03,54.67,True
517,2024-10,DST_0178,Dang,Gujarat,119,110,217,54.84,50.69,True
606,2024-10,DST_0267,Dholpur,Rajasthan,437,1723,1319,33.13,130.63,True


In [30]:
# === PHASE 4: CLEAN DISTRIBUTION ANALYSIS COPY ===

distribution_analysis = distribution.copy()

# Keep only records without impossible distribution values
distribution_analysis = distribution_analysis[
    ~distribution_analysis["distribution_data_quality_issue"]
].copy()

print("=== DISTRIBUTION ANALYSIS DATA ===")

print("Original rows:", len(distribution))
print("Flagged rows excluded:", distribution["distribution_data_quality_issue"].sum())
print("Rows retained:", len(distribution_analysis))

print(
    "Districts retained:",
    distribution_analysis["District_ID"].nunique()
)

print(
    "District × Audit Cycle:",
    distribution_analysis[
        ["District_ID", "Audit_Cycle"]
    ].drop_duplicates().shape[0]
)

print("\nRemaining logical violations:")

print(
    "Kestrel > category:",
    (
        distribution_analysis["Outlets_Stocking_Kestrel"]
        > distribution_analysis["Outlets_Stocking_Category"]
    ).sum()
)

print(
    "Category > universe:",
    (
        distribution_analysis["Outlets_Stocking_Category"]
        > distribution_analysis["Estimated_Total_Universe"]
    ).sum()
)

print(
    "Kestrel > universe:",
    (
        distribution_analysis["Outlets_Stocking_Kestrel"]
        > distribution_analysis["Estimated_Total_Universe"]
    ).sum()
)

=== DISTRIBUTION ANALYSIS DATA ===
Original rows: 8160
Flagged rows excluded: 159
Rows retained: 8001
Districts retained: 340
District × Audit Cycle: 8001

Remaining logical violations:
Kestrel > category: 0
Category > universe: 0
Kestrel > universe: 0


In [31]:
# === PHASE 4: DISTRIBUTION GAP ===

# Reload the finalized processed distribution dataset
distribution = pd.read_csv(
    PROCESSED_DATA / "distribution_processed.csv"
)

# Keep only valid distribution observations
distribution_valid = distribution[
    ~distribution["distribution_data_quality_issue"]
].copy()

# Calculate monthly Distribution Gap
distribution_valid["Distribution_Gap"] = (
    1
    - (
        distribution_valid["Outlets_Stocking_Kestrel"]
        / distribution_valid["Outlets_Stocking_Category"]
    )
)

print("=== DISTRIBUTION GAP ===")

print("Original distribution rows:", len(distribution))
print("Flagged rows excluded:", distribution[
    "distribution_data_quality_issue"
].sum())
print("Valid rows:", len(distribution_valid))

print(
    "Districts:",
    distribution_valid["District_ID"].nunique()
)

print(
    "District × Audit Cycle:",
    distribution_valid[
        ["District_ID", "Audit_Cycle"]
    ].drop_duplicates().shape[0]
)

print("\nDistribution Gap range:")
print(
    round(distribution_valid["Distribution_Gap"].min(), 4),
    "to",
    round(distribution_valid["Distribution_Gap"].max(), 4)
)

# Aggregate monthly gaps to district level
distribution_gap = (
    distribution_valid
    .groupby("District_ID", as_index=False)
    .agg(
        Mean_Distribution_Gap=("Distribution_Gap", "mean"),
        Valid_Audit_Cycles=("Audit_Cycle", "nunique")
    )
)

print("\n=== DISTRICT-LEVEL DISTRIBUTION GAP ===")

print(
    "Districts with valid distribution data:",
    len(distribution_gap)
)

print(
    "Minimum valid audit cycles:",
    distribution_gap["Valid_Audit_Cycles"].min()
)

print(
    "Maximum valid audit cycles:",
    distribution_gap["Valid_Audit_Cycles"].max()
)

display(
    distribution_gap
    .sort_values(
        "Mean_Distribution_Gap",
        ascending=False
    )
    .head(20)
)

=== DISTRIBUTION GAP ===
Original distribution rows: 8160
Flagged rows excluded: 159
Valid rows: 8001
Districts: 340
District × Audit Cycle: 8001

Distribution Gap range:
0.0 to 0.9625

=== DISTRICT-LEVEL DISTRIBUTION GAP ===
Districts with valid distribution data: 340
Minimum valid audit cycles: 14
Maximum valid audit cycles: 24


,District_ID,Mean_Distribution_Gap,Valid_Audit_Cycles
300,DST_0301,0.933118,24
39,DST_0040,0.931602,24
6,DST_0007,0.930945,24
38,DST_0039,0.930687,24
152,DST_0153,0.930514,24
270,DST_0271,0.930131,24
245,DST_0246,0.929079,24
328,DST_0329,0.928676,24
210,DST_0211,0.928409,24
243,DST_0244,0.927974,23


In [32]:
# === PHASE 4: DISTRIBUTION GAP COMPLETENESS CHECK ===

print("=== DISTRIBUTION GAP COMPLETENESS ===")

print(
    "Districts:",
    distribution_gap["District_ID"].nunique()
)

print(
    "Minimum valid audit cycles:",
    distribution_gap["Valid_Audit_Cycles"].min()
)

print(
    "Maximum valid audit cycles:",
    distribution_gap["Valid_Audit_Cycles"].max()
)

print("\n=== VALID AUDIT CYCLE DISTRIBUTION ===")

display(
    distribution_gap["Valid_Audit_Cycles"]
    .value_counts()
    .sort_index()
    .to_frame("Districts")
)

print("\n=== DISTRIBUTION GAP VALIDATION ===")

print(
    "Missing Mean Distribution Gap:",
    distribution_gap["Mean_Distribution_Gap"].isna().sum()
)

print(
    "Gap below 0:",
    (distribution_gap["Mean_Distribution_Gap"] < 0).sum()
)

print(
    "Gap above 1:",
    (distribution_gap["Mean_Distribution_Gap"] > 1).sum()
)

print(
    "Category outlets equal to zero:",
    (
        distribution_valid["Outlets_Stocking_Category"] == 0
    ).sum()
)

if (
    distribution_gap["Mean_Distribution_Gap"].isna().sum() == 0
    and (distribution_gap["Mean_Distribution_Gap"] < 0).sum() == 0
    and (distribution_gap["Mean_Distribution_Gap"] > 1).sum() == 0
    and (
        distribution_valid["Outlets_Stocking_Category"] == 0
    ).sum() == 0
):
    print("\n✓ DISTRIBUTION GAP VALIDATION PASSED")
else:
    print("\n⚠ CHECK REQUIRED")

=== DISTRIBUTION GAP COMPLETENESS ===
Districts: 340
Minimum valid audit cycles: 14
Maximum valid audit cycles: 24

=== VALID AUDIT CYCLE DISTRIBUTION ===


,Districts
Valid_Audit_Cycles,
14,3
15,2
16,1
17,2
18,2
19,4
20,1
22,4
23,45



=== DISTRIBUTION GAP VALIDATION ===
Missing Mean Distribution Gap: 0
Gap below 0: 0
Gap above 1: 0
Category outlets equal to zero: 0

✓ DISTRIBUTION GAP VALIDATION PASSED


In [33]:
# === PHASE 4: DEMAND + DISTRIBUTION MASTER ===

phase4_master = demand_signal.merge(
    distribution_gap[
        [
            "District_ID",
            "Mean_Distribution_Gap",
            "Valid_Audit_Cycles"
        ]
    ],
    left_on="district_id",
    right_on="District_ID",
    how="left",
    validate="one_to_one"
)

print("=== PHASE 4 MASTER VALIDATION ===")

print(
    "Districts:",
    phase4_master["district_id"].nunique()
)

print(
    "Missing Demand Signal:",
    phase4_master["Demand_Signal"].isna().sum()
)

print(
    "Missing Distribution Gap:",
    phase4_master["Mean_Distribution_Gap"].isna().sum()
)

print(
    "Missing valid-cycle count:",
    phase4_master["Valid_Audit_Cycles"].isna().sum()
)

print("\n=== PHASE 4 COMPONENT RANGES ===")

print(
    "Demand Signal:",
    round(phase4_master["Demand_Signal"].min(), 4),
    "to",
    round(phase4_master["Demand_Signal"].max(), 4)
)

print(
    "Distribution Gap:",
    round(phase4_master["Mean_Distribution_Gap"].min(), 4),
    "to",
    round(phase4_master["Mean_Distribution_Gap"].max(), 4)
)

print("\n=== TOP HIGH-DEMAND + HIGH-DISTRIBUTION-GAP DISTRICTS ===")

display(
    phase4_master[
        [
            "district_id",
            "Demand_Signal",
            "Mean_Distribution_Gap",
            "Valid_Audit_Cycles"
        ]
    ]
    .sort_values(
        [
            "Demand_Signal",
            "Mean_Distribution_Gap"
        ],
        ascending=False
    )
    .head(20)
)

=== PHASE 4 MASTER VALIDATION ===
Districts: 340
Missing Demand Signal: 0
Missing Distribution Gap: 0
Missing valid-cycle count: 0

=== PHASE 4 COMPONENT RANGES ===
Demand Signal: 0.0088 to 0.9686
Distribution Gap: 0.0879 to 0.9331

=== TOP HIGH-DEMAND + HIGH-DISTRIBUTION-GAP DISTRICTS ===


,district_id,Demand_Signal,Mean_Distribution_Gap,Valid_Audit_Cycles
240,DST_0241,0.968627,0.667486,24
271,DST_0272,0.958333,0.667741,24
99,DST_0100,0.956863,0.354800,23
272,DST_0273,0.954902,0.656122,24
212,DST_0213,0.952451,0.658044,24
191,DST_0192,0.944608,0.924601,24
39,DST_0040,0.935294,0.931602,24
242,DST_0243,0.933333,0.656711,24
329,DST_0330,0.930392,0.662417,24
296,DST_0297,0.929902,0.668104,24


In [34]:
# === SAVE PHASE 4 MASTER ===

OUTPUT_DATA.mkdir(parents=True, exist_ok=True)

phase4_master.to_csv(
    OUTPUT_DATA / "phase4_demand_distribution.csv",
    index=False
)

print("=== PHASE 4 MASTER SAVED ===")
print("Rows:", len(phase4_master))
print("Columns:", len(phase4_master.columns))
print(
    "File:",
    OUTPUT_DATA / "phase4_demand_distribution.csv"
)

=== PHASE 4 MASTER SAVED ===
Rows: 340
Columns: 11
File: D:\sivahitesh\Desktop\Tail_project\data\outputs\phase4_demand_distribution.csv


In [35]:
# === PHASE 4: COMPETITOR DATA STRUCTURE ===

competitor = pd.read_csv(
    PROCESSED_DATA / "competitor_processed.csv"
)

print("=== COMPETITOR DATA STRUCTURE ===")

print("Rows:", len(competitor))
print("Columns:")
print(competitor.columns.tolist())

print("\nData types:")
print(competitor.dtypes)

print("\nFirst 10 rows:")
display(competitor.head(10))

=== COMPETITOR DATA STRUCTURE ===
Rows: 40800
Columns:
['audit_date', 'month_id', 'district_id', 'district_name', 'state', 'competitor_brand', 'numeric_distribution_pct', 'weighted_distribution_pct', 'stocking_outlet_count', 'exclusive_branded_chillers_deployed', 'promotional_intensity_score_1_to_10', 'active_retail_scheme', 'audit_date_parsed', 'audit_date_is_future', 'district_id_original', 'district_id_normalized', 'district_name_standardized']

Data types:
audit_date                              object
month_id                                object
district_id                             object
district_name                           object
state                                   object
competitor_brand                        object
numeric_distribution_pct               float64
weighted_distribution_pct              float64
stocking_outlet_count                    int64
exclusive_branded_chillers_deployed      int64
promotional_intensity_score_1_to_10    float64
active_retail_sche

,audit_date,month_id,district_id,district_name,state,competitor_brand,numeric_distribution_pct,weighted_distribution_pct,stocking_outlet_count,exclusive_branded_chillers_deployed,promotional_intensity_score_1_to_10,active_retail_scheme,audit_date_parsed,audit_date_is_future,district_id_original,district_id_normalized,district_name_standardized
0,2026-07-15T00:00:00+05:30,2026-07,DST_0043,Shivamogga,Karnataka,Torque,47.71,54.00,1090,104,4.69,NaN,2026-07-14 18:30:00+00:00,False,DST_0043,DST_0043,Shivamogga
1,2025/03/15,2025-03,DST_0292,Sri Sathya Sai,Andhra Pradesh,BullForce,38.60,58.47,786,78,4.31,NaN,2025-03-15 00:00:00+00:00,False,DST_0292,DST_0292,Sri Sathya Sai
2,2025-09-09T00:00:00+05:30,2025-09,DST_0024,Dhule,Maharashtra,Torque,45.81,49.59,1119,77,4.01,Seasonal_Discount,2025-09-08 18:30:00+00:00,False,DST_0024,DST_0024,Dhule
3,2026/05/18,2026-05,DST_0092,Thoothukudi,Tamil Nadu,PulseDrop,38.34,58.43,1006,156,2.53,NaN,2026-05-18 00:00:00+00:00,False,DST_0092,DST_0092,Thoothukudi
4,1784448540000,2026-07,DST_0038,Mysuru,Karnataka,Apex Energy,92.73,96.35,3823,2302,9.71,Active_BOGO_Scheme,2026-07-19 08:09:00+00:00,False,DST_0038,DST_0038,Mysuru
5,2025/10/15,2025-10,DST_0145,Gonda,Uttar Pradesh,Apex Energy,30.97,60.32,954,103,3.40,NaN,2025-10-15 00:00:00+00:00,False,DST_0145,DST_0145,Gonda
6,2025/09/18,2025-09,0128,Firozabad,Uttar Pradesh,Torque,45.26,55.56,1442,107,4.10,NaN,2025-09-18 00:00:00+00:00,False,0128,DST_0128,Firozabad
7,2026/08/08,2026-08,DST_0320,Narmadapuram,Madhya Pradesh,Voltix,30.71,61.10,470,64,5.27,NaN,2026-08-08 00:00:00+00:00,False,DST_0320,DST_0320,Narmadapuram
8,17-10-2024,2024-10,DST_0130,Shahjahanpur,Uttar Pradesh,Voltix,39.67,61.02,1288,130,4.74,Seasonal_Discount,2024-10-17 00:00:00+00:00,False,DST_0130,DST_0130,Shahjahanpur
9,1770114780000,2026-02,DST_0154,Rajkot,Gujarat,Voltix,55.21,58.34,3385,565,5.05,Special_Kirana_Margin,2026-02-03 10:33:00+00:00,False,DST_0154,DST_0154,Rajkot


In [36]:
# === PHASE 4: COMPETITOR CATEGORY VOLUME ===

# Use the processed Retail Panel only
retail_comp = retail_panel.copy()

# Identify brand from the SKU prefix
brand_map = {
    "APX": "Apex Energy",
    "BLF": "BullForce",
    "PLS": "PulseDrop",
    "TRQ": "Torque",
    "VLT": "Voltix",
    "KES": "Kestrel"
}

retail_comp["Brand"] = (
    retail_comp["sku_id"]
    .astype(str)
    .str.split("_")
    .str[0]
    .map(brand_map)
)

# Check that every SKU received a brand
print("=== BRAND IDENTIFICATION ===")

print(
    "Unmapped SKUs:",
    retail_comp["Brand"].isna().sum()
)

print("\nBrand counts:")
display(
    retail_comp["Brand"]
    .value_counts(dropna=False)
)

# Keep only competitor brands
competitor_volume = retail_comp[
    retail_comp["Brand"].notna()
    & (retail_comp["Brand"] != "Kestrel")
].copy()

# Calculate projected category volume
competitor_volume["Projected_Units"] = (
    competitor_volume["units_sold"]
    * competitor_volume["projection_weight"]
)

# Aggregate competitor volume by district and brand
competitor_brand_volume = (
    competitor_volume
    .groupby(
        ["district_id", "Brand"],
        as_index=False
    )
    .agg(
        Competitor_Volume=("Projected_Units", "sum")
    )
)

print("\n=== COMPETITOR BRAND VOLUME ===")

print(
    "Districts:",
    competitor_brand_volume["district_id"].nunique()
)

print(
    "Competitor brands:",
    competitor_brand_volume["Brand"].nunique()
)

print(
    "Total district × competitor-brand records:",
    len(competitor_brand_volume)
)

display(
    competitor_brand_volume
    .sort_values(
        "Competitor_Volume",
        ascending=False
    )
    .head(20)
)

=== BRAND IDENTIFICATION ===
Unmapped SKUs: 0

Brand counts:


Brand
Apex Energy    417947
Voltix         348461
Kestrel        165912
BullForce      156760
Torque         155769
PulseDrop      155444
Name: count, dtype: int64


=== COMPETITOR BRAND VOLUME ===
Districts: 340
Competitor brands: 5
Total district × competitor-brand records: 1700


,district_id,Brand,Competitor_Volume
755,DST_0152,Apex Energy,3.950005e+08
759,DST_0152,Voltix,2.494248e+08
1645,DST_0330,Apex Energy,1.659533e+08
1480,DST_0297,Apex Energy,1.555510e+08
325,DST_0066,Apex Energy,1.539190e+08
1484,DST_0297,Voltix,1.462811e+08
939,DST_0188,Voltix,1.449279e+08
935,DST_0188,Apex Energy,1.394239e+08
329,DST_0066,Voltix,1.315542e+08
575,DST_0116,Apex Energy,1.273515e+08


In [37]:
# === PHASE 4: COMPETITIVE INTENSITY ===

# Total competitor volume in each district
district_competitor_total = (
    competitor_brand_volume
    .groupby("district_id", as_index=False)
    .agg(
        Total_Competitor_Volume=("Competitor_Volume", "sum")
    )
)

# Calculate each competitor's share within the district
competitor_brand_volume = competitor_brand_volume.merge(
    district_competitor_total,
    on="district_id",
    how="left",
    validate="many_to_one"
)

competitor_brand_volume["Competitor_Share"] = (
    competitor_brand_volume["Competitor_Volume"]
    / competitor_brand_volume["Total_Competitor_Volume"]
)

# Find the two strongest competitors in each district
top_two_competitors = (
    competitor_brand_volume
    .sort_values(
        ["district_id", "Competitor_Share"],
        ascending=[True, False]
    )
    .groupby("district_id")
    .head(2)
)

# Sum the shares of the top two competitors
competitive_intensity = (
    top_two_competitors
    .groupby("district_id", as_index=False)
    .agg(
        Competitive_Intensity=("Competitor_Share", "sum")
    )
)

print("=== COMPETITIVE INTENSITY ===")
print("Districts:", competitive_intensity["district_id"].nunique())
print(
    "Missing Competitive Intensity:",
    competitive_intensity["Competitive_Intensity"].isna().sum()
)
print(
    "Range:",
    round(competitive_intensity["Competitive_Intensity"].min(), 4),
    "to",
    round(competitive_intensity["Competitive_Intensity"].max(), 4)
)

print("\n=== TOP COMPETITIVE INTENSITY DISTRICTS ===")

display(
    competitive_intensity
    .sort_values("Competitive_Intensity", ascending=False)
    .head(20)
)

=== COMPETITIVE INTENSITY ===
Districts: 340
Missing Competitive Intensity: 0
Range: 0.5045 to 0.9011

=== TOP COMPETITIVE INTENSITY DISTRICTS ===


,district_id,Competitive_Intensity
237,DST_0238,0.901083
151,DST_0152,0.900793
271,DST_0272,0.891312
187,DST_0188,0.889767
329,DST_0330,0.887726
272,DST_0273,0.884283
112,DST_0113,0.880625
242,DST_0243,0.874786
115,DST_0116,0.874350
212,DST_0213,0.874137


In [45]:
# === PHASE 4: LOAD CLEANED COVERAGE DATA ===

PROCESSED_DATA = Path(
    r"D:\sivahitesh\Desktop\Tail_project\data\processed"
)

panel_coverage = pd.read_csv(
    PROCESSED_DATA / "panel_coverage_processed.csv"
)

district_master = pd.read_csv(
    PROCESSED_DATA / "district_master_processed.csv"
)

print("=== CLEANED COVERAGE DATA LOADED ===")
print("Panel Coverage:", panel_coverage.shape)
print("District Master:", district_master.shape)

print("\nPanel Coverage columns:")
print(panel_coverage.columns.tolist())

print("\nNormalized district IDs:",
      panel_coverage["District_Ref_ID_normalized"].nunique())

print("District Master IDs:",
      district_master["District_Code"].nunique())

=== CLEANED COVERAGE DATA LOADED ===
Panel Coverage: (8160, 9)
District Master: (340, 9)

Panel Coverage columns:
['District_Ref_ID', 'Month', 'District_Name', 'State', 'Sampled_Outlets', 'Estimated_Total_Outlets', 'Panel_Coverage_Pct', 'Audit_Reliability_Tier', 'District_Ref_ID_normalized']

Normalized district IDs: 340
District Master IDs: 340


In [48]:
import numpy as np

district_coverage["Coverage_Status"] = np.where(
    district_coverage["Mean_Panel_Coverage_Pct"] < 60,
    "UNKNOWN",
    "RELIABLE"
)

print("=== PANEL COVERAGE STATUS ===")
print("Districts:", len(district_coverage))
print(
    "Missing coverage:",
    district_coverage["Mean_Panel_Coverage_Pct"].isna().sum()
)
print(
    "UNKNOWN:",
    (district_coverage["Coverage_Status"] == "UNKNOWN").sum()
)
print(
    "RELIABLE:",
    (district_coverage["Coverage_Status"] == "RELIABLE").sum()
)

display(
    district_coverage[
        [
            "District_Code",
            "District_Name",
            "Mean_Panel_Coverage_Pct",
            "Coverage_Status"
        ]
    ].head(10)
)

=== PANEL COVERAGE STATUS ===
Districts: 340
Missing coverage: 0
UNKNOWN: 257
RELIABLE: 83


,District_Code,District_Name,Mean_Panel_Coverage_Pct,Coverage_Status
0,DST_0001,Mumbai Suburban,85.037917,RELIABLE
1,DST_0002,Mumbai City,82.070417,RELIABLE
2,DST_0003,Pune,83.369167,RELIABLE
3,DST_0004,Thane,41.783333,UNKNOWN
4,DST_0005,Palghar,15.612500,UNKNOWN
5,DST_0006,Nagpur,55.080833,UNKNOWN
6,DST_0007,Nashik,72.070000,RELIABLE
7,DST_0008,Aurangabad,57.322500,UNKNOWN
8,DST_0009,Kolhapur,68.176250,RELIABLE
9,DST_0010,Solapur,36.100000,UNKNOWN


In [51]:
# === PHASE 4: ADD PANEL COVERAGE STATUS ===

phase4_market = phase4_market.merge(
    district_coverage[
        [
            "District_Code",
            "Mean_Panel_Coverage_Pct",
            "Coverage_Status"
        ]
    ],
    left_on="district_id",
    right_on="District_Code",
    how="left",
    validate="one_to_one"
)

print("=== PHASE 4 MARKET + COVERAGE ===")
print("Districts:", len(phase4_market))
print(
    "Missing coverage:",
    phase4_market["Mean_Panel_Coverage_Pct"].isna().sum()
)
print(
    "Missing coverage status:",
    phase4_market["Coverage_Status"].isna().sum()
)

display(
    phase4_market[
        [
            "district_id",
            "Demand_Signal",
            "Mean_Distribution_Gap",
            "Mean_Panel_Coverage_Pct",
            "Coverage_Status"
        ]
    ].head(15)
)

=== PHASE 4 MARKET + COVERAGE ===
Districts: 340
Missing coverage: 0
Missing coverage status: 0


,district_id,Demand_Signal,Mean_Distribution_Gap,Mean_Panel_Coverage_Pct,Coverage_Status
0,DST_0001,0.495098,0.365108,85.037917,RELIABLE
1,DST_0002,0.926471,0.378500,82.070417,RELIABLE
2,DST_0003,0.633824,0.369119,83.369167,RELIABLE
3,DST_0004,0.453431,0.438038,41.783333,UNKNOWN
4,DST_0005,0.498039,0.576765,15.612500,UNKNOWN
5,DST_0006,0.513725,0.465464,55.080833,UNKNOWN
6,DST_0007,0.515196,0.930945,72.070000,RELIABLE
7,DST_0008,0.873529,0.926553,57.322500,UNKNOWN
8,DST_0009,0.811765,0.672621,68.176250,RELIABLE
9,DST_0010,0.284314,0.452912,36.100000,UNKNOWN


In [52]:
# === PHASE 4: MARKET OPPORTUNITY CLASSIFICATION ===

phase4_market["Market_Opportunity"] = np.where(
    (phase4_market["Demand_Signal"] >= 0.60) &
    (phase4_market["Mean_Distribution_Gap"] >= 0.60),
    "HIGH_OPPORTUNITY",
    "OTHER"
)

print("=== PHASE 4 MARKET OPPORTUNITY ===")
print(
    phase4_market["Market_Opportunity"].value_counts()
)

print("\n=== HIGH OPPORTUNITY DISTRICTS ===")

high_opportunity = (
    phase4_market[
        phase4_market["Market_Opportunity"] == "HIGH_OPPORTUNITY"
    ]
    .sort_values(
        ["Demand_Signal", "Mean_Distribution_Gap"],
        ascending=False
    )
    .reset_index(drop=True)
)

print("High-opportunity districts:", len(high_opportunity))

display(
    high_opportunity[
        [
            "district_id",
            "Demand_Signal",
            "Mean_Distribution_Gap",
            "Mean_Panel_Coverage_Pct",
            "Coverage_Status",
            "Market_Opportunity"
        ]
    ].head(20)
)

=== PHASE 4 MARKET OPPORTUNITY ===
Market_Opportunity
OTHER               312
HIGH_OPPORTUNITY     28
Name: count, dtype: int64

=== HIGH OPPORTUNITY DISTRICTS ===
High-opportunity districts: 28


,district_id,Demand_Signal,Mean_Distribution_Gap,Mean_Panel_Coverage_Pct,Coverage_Status,Market_Opportunity
0,DST_0241,0.968627,0.667486,68.012917,RELIABLE,HIGH_OPPORTUNITY
1,DST_0272,0.958333,0.667741,65.166250,RELIABLE,HIGH_OPPORTUNITY
2,DST_0273,0.954902,0.656122,73.240417,RELIABLE,HIGH_OPPORTUNITY
3,DST_0213,0.952451,0.658044,68.736667,RELIABLE,HIGH_OPPORTUNITY
4,DST_0192,0.944608,0.924601,60.542917,RELIABLE,HIGH_OPPORTUNITY
5,DST_0040,0.935294,0.931602,66.228750,RELIABLE,HIGH_OPPORTUNITY
6,DST_0243,0.933333,0.656711,70.208333,RELIABLE,HIGH_OPPORTUNITY
7,DST_0330,0.930392,0.662417,76.819167,RELIABLE,HIGH_OPPORTUNITY
8,DST_0297,0.929902,0.668104,68.488750,RELIABLE,HIGH_OPPORTUNITY
9,DST_0038,0.928922,0.683777,72.087083,RELIABLE,HIGH_OPPORTUNITY


In [54]:
# === PHASE 4: SAVE MARKET OPPORTUNITY OUTPUT ===

OUTPUT_DATA.mkdir(parents=True, exist_ok=True)

phase4_market.to_csv(
    OUTPUT_DATA / "market_opportunity.csv",
    index=False
)

high_opportunity.to_csv(
    OUTPUT_DATA / "high_opportunity_districts.csv",
    index=False
)

print("=== PHASE 4 OUTPUTS SAVED ===")
print("market_opportunity.csv")
print("high_opportunity_districts.csv")

print("\nRows saved:")
print("Complete market table:", len(phase4_market))
print("High-opportunity districts:", len(high_opportunity))

=== PHASE 4 OUTPUTS SAVED ===
market_opportunity.csv
high_opportunity_districts.csv

Rows saved:
Complete market table: 340
High-opportunity districts: 28


In [55]:
# === PHASE 4: LOAD COMPETITIVE DATA ===

competitor = pd.read_csv(
    PROCESSED_DATA / "competitor_processed.csv"
)

print("=== COMPETITOR DATA LOADED ===")
print("Rows:", len(competitor))
print("Columns:", len(competitor.columns))
print("Unique districts:", competitor["district_id_normalized"].nunique())
print("Unique competitor brands:", competitor["competitor_brand"].nunique())

print("\nCompetitor brands:")
print(competitor["competitor_brand"].value_counts())

print("\nFirst 5 rows:")
display(competitor.head())

=== COMPETITOR DATA LOADED ===
Rows: 40800
Columns: 17
Unique districts: 340
Unique competitor brands: 5

Competitor brands:
competitor_brand
Torque         8160
BullForce      8160
PulseDrop      8160
Apex Energy    8160
Voltix         8160
Name: count, dtype: int64

First 5 rows:


,audit_date,month_id,district_id,district_name,state,competitor_brand,numeric_distribution_pct,weighted_distribution_pct,stocking_outlet_count,exclusive_branded_chillers_deployed,promotional_intensity_score_1_to_10,active_retail_scheme,audit_date_parsed,audit_date_is_future,district_id_original,district_id_normalized,district_name_standardized
0,2026-07-15T00:00:00+05:30,2026-07,DST_0043,Shivamogga,Karnataka,Torque,47.71,54.00,1090,104,4.69,NaN,2026-07-14 18:30:00+00:00,False,DST_0043,DST_0043,Shivamogga
1,2025/03/15,2025-03,DST_0292,Sri Sathya Sai,Andhra Pradesh,BullForce,38.60,58.47,786,78,4.31,NaN,2025-03-15 00:00:00+00:00,False,DST_0292,DST_0292,Sri Sathya Sai
2,2025-09-09T00:00:00+05:30,2025-09,DST_0024,Dhule,Maharashtra,Torque,45.81,49.59,1119,77,4.01,Seasonal_Discount,2025-09-08 18:30:00+00:00,False,DST_0024,DST_0024,Dhule
3,2026/05/18,2026-05,DST_0092,Thoothukudi,Tamil Nadu,PulseDrop,38.34,58.43,1006,156,2.53,NaN,2026-05-18 00:00:00+00:00,False,DST_0092,DST_0092,Thoothukudi
4,1784448540000,2026-07,DST_0038,Mysuru,Karnataka,Apex Energy,92.73,96.35,3823,2302,9.71,Active_BOGO_Scheme,2026-07-19 08:09:00+00:00,False,DST_0038,DST_0038,Mysuru


In [56]:
# === PHASE 4: DISTRICT-LEVEL COMPETITIVE AGGREGATION ===

competitor_district = (
    competitor
    .groupby("district_id_normalized", as_index=False)
    .agg(
        Competitor_Volume=("stocking_outlet_count", "sum"),
        Mean_Numeric_Distribution=("numeric_distribution_pct", "mean"),
        Mean_Weighted_Distribution=("weighted_distribution_pct", "mean"),
        Mean_Promotional_Intensity=("promotional_intensity_score_1_to_10", "mean"),
        Total_Exclusive_Chillers=("exclusive_branded_chillers_deployed", "sum")
    )
)

print("=== DISTRICT-LEVEL COMPETITIVE AGGREGATION ===")
print("Districts:", len(competitor_district))
print(
    "Missing district IDs:",
    competitor_district["district_id_normalized"].isna().sum()
)

print("\nCompetitive volume range:")
print(
    competitor_district["Competitor_Volume"].min(),
    "to",
    competitor_district["Competitor_Volume"].max()
)

print("\nFirst 10 districts:")
display(competitor_district.head(10))

=== DISTRICT-LEVEL COMPETITIVE AGGREGATION ===
Districts: 340
Missing district IDs: 0

Competitive volume range:
4225 to 1757217

First 10 districts:


,district_id_normalized,Competitor_Volume,Mean_Numeric_Distribution,Mean_Weighted_Distribution,Mean_Promotional_Intensity,Total_Exclusive_Chillers
0,DST_0001,1757217,79.156667,82.865083,7.195500,707449
1,DST_0002,643168,78.977833,82.871250,7.278167,249162
2,DST_0003,1461413,78.089667,82.987750,7.232000,579980
3,DST_0004,724461,39.839000,46.318250,4.016667,87479
4,DST_0005,214163,40.562417,46.991667,3.939167,26107
5,DST_0006,395019,40.381500,48.296167,4.089333,49619
6,DST_0007,506826,49.334583,57.846000,4.720583,85568
7,DST_0008,309578,49.234500,56.412000,4.608250,54205
8,DST_0009,333569,57.477417,65.479417,5.916500,130057
9,DST_0010,252384,39.045167,47.345250,3.965083,31882


In [57]:
# === PHASE 4: NORMALIZE COMPETITIVE COMPONENTS ===

competitive_components = competitor_district.copy()

components = [
    "Competitor_Volume",
    "Mean_Numeric_Distribution",
    "Mean_Weighted_Distribution",
    "Mean_Promotional_Intensity",
    "Total_Exclusive_Chillers"
]

for col in components:
    min_val = competitive_components[col].min()
    max_val = competitive_components[col].max()

    competitive_components[col + "_Score"] = (
        (competitive_components[col] - min_val)
        / (max_val - min_val)
    )

print("=== NORMALIZED COMPETITIVE COMPONENTS ===")
print("Districts:", len(competitive_components))

for col in components:
    score_col = col + "_Score"
    print(
        f"{score_col}: "
        f"{competitive_components[score_col].min():.4f} "
        f"to {competitive_components[score_col].max():.4f}"
    )

display(
    competitive_components[
        ["district_id_normalized"] +
        [col + "_Score" for col in components]
    ].head(10)
)

=== NORMALIZED COMPETITIVE COMPONENTS ===
Districts: 340
Competitor_Volume_Score: 0.0000 to 1.0000
Mean_Numeric_Distribution_Score: 0.0000 to 1.0000
Mean_Weighted_Distribution_Score: 0.0000 to 1.0000
Mean_Promotional_Intensity_Score: 0.0000 to 1.0000
Total_Exclusive_Chillers_Score: 0.0000 to 1.0000


,district_id_normalized,Competitor_Volume_Score,Mean_Numeric_Distribution_Score,Mean_Weighted_Distribution_Score,Mean_Promotional_Intensity_Score,Total_Exclusive_Chillers_Score
0,DST_0001,1.000000,0.995141,0.986181,0.970914,0.992091
1,DST_0002,0.364487,0.992302,0.986277,0.986477,0.349180
2,DST_0003,0.831258,0.978205,0.988084,0.977786,0.813270
3,DST_0004,0.410861,0.371089,0.419309,0.372480,0.122361
4,DST_0005,0.119760,0.382571,0.429754,0.357890,0.036265
5,DST_0006,0.222930,0.379699,0.449988,0.386160,0.069249
6,DST_0007,0.286710,0.521803,0.598114,0.504997,0.119681
7,DST_0008,0.174190,0.520214,0.575871,0.483849,0.075683
8,DST_0009,0.187875,0.651046,0.716515,0.730135,0.182092
9,DST_0010,0.141563,0.358489,0.435238,0.362769,0.044367


In [58]:
# === PHASE 4: COMPETITIVE INTENSITY ===

competitive_components["Competitive_Intensity"] = (
    competitive_components["Competitor_Volume_Score"] +
    competitive_components["Mean_Numeric_Distribution_Score"] +
    competitive_components["Mean_Weighted_Distribution_Score"] +
    competitive_components["Mean_Promotional_Intensity_Score"] +
    competitive_components["Total_Exclusive_Chillers_Score"]
) / 5

print("=== COMPETITIVE INTENSITY ===")
print("Districts:", len(competitive_components))
print(
    "Missing Competitive Intensity:",
    competitive_components["Competitive_Intensity"].isna().sum()
)
print(
    "Range:",
    f"{competitive_components['Competitive_Intensity'].min():.4f}",
    "to",
    f"{competitive_components['Competitive_Intensity'].max():.4f}"
)

print("\n=== TOP COMPETITIVE INTENSITY DISTRICTS ===")

display(
    competitive_components[
        ["district_id_normalized", "Competitive_Intensity"]
    ]
    .sort_values("Competitive_Intensity", ascending=False)
    .head(20)
)

=== COMPETITIVE INTENSITY ===
Districts: 340
Missing Competitive Intensity: 0
Range: 0.0075 to 0.9889

=== TOP COMPETITIVE INTENSITY DISTRICTS ===


,district_id_normalized,Competitive_Intensity
0,DST_0001,0.988865
35,DST_0036,0.986285
64,DST_0065,0.928509
2,DST_0003,0.917720
150,DST_0151,0.901720
183,DST_0184,0.808999
206,DST_0207,0.775953
1,DST_0002,0.735744
105,DST_0106,0.715503
99,DST_0100,0.618754


In [59]:
# === PHASE 4: SAVE COMPETITIVE ANALYSIS ===

OUTPUT_DATA.mkdir(parents=True, exist_ok=True)

competitive_components.to_csv(
    OUTPUT_DATA / "competitive_intensity_analysis.csv",
    index=False
)

top_competitive_districts = (
    competitive_components[
        ["district_id_normalized", "Competitive_Intensity"]
    ]
    .sort_values("Competitive_Intensity", ascending=False)
    .reset_index(drop=True)
)

top_competitive_districts.to_csv(
    OUTPUT_DATA / "top_competitive_intensity_districts.csv",
    index=False
)

print("=== COMPETITIVE ANALYSIS SAVED ===")
print("Complete competitive table:", len(competitive_components))
print("Competitive districts:", competitive_components["district_id_normalized"].nunique())
print("Files saved:")
print("✓ competitive_intensity_analysis.csv")
print("✓ top_competitive_intensity_districts.csv")

=== COMPETITIVE ANALYSIS SAVED ===
Complete competitive table: 340
Competitive districts: 340
Files saved:
✓ competitive_intensity_analysis.csv
✓ top_competitive_intensity_districts.csv


In [62]:
# === PHASE 4: FIX DUPLICATE COMPETITIVE COLUMNS ===

# Keep the newly calculated competitive intensity
if "Competitive_Intensity_y" in phase4_final.columns:
    phase4_final["Competitive_Intensity"] = phase4_final[
        "Competitive_Intensity_y"
    ]
elif "Competitive_Intensity_x" in phase4_final.columns:
    phase4_final["Competitive_Intensity"] = phase4_final[
        "Competitive_Intensity_x"
    ]

# Remove duplicate versions
drop_cols = [
    col for col in [
        "Competitive_Intensity_x",
        "Competitive_Intensity_y"
    ]
    if col in phase4_final.columns
]

phase4_final = phase4_final.drop(columns=drop_cols)

print("=== PHASE 4 COMPETITIVE COLUMN FIXED ===")
print(
    "Competitive Intensity column exists:",
    "Competitive_Intensity" in phase4_final.columns
)
print(
    "Missing Competitive Intensity:",
    phase4_final["Competitive_Intensity"].isna().sum()
)

display(
    phase4_final[
        [
            "district_id",
            "Demand_Signal",
            "Mean_Distribution_Gap",
            "Competitive_Intensity",
            "Mean_Panel_Coverage_Pct",
            "Coverage_Status"
        ]
    ].head(20)
)


=== PHASE 4 COMPETITIVE COLUMN FIXED ===
Competitive Intensity column exists: True
Missing Competitive Intensity: 0


,district_id,Demand_Signal,Mean_Distribution_Gap,Competitive_Intensity,Mean_Panel_Coverage_Pct,Coverage_Status
0,DST_0001,0.495098,0.365108,0.988865,85.037917,RELIABLE
1,DST_0002,0.926471,0.378500,0.735744,82.070417,RELIABLE
2,DST_0003,0.633824,0.369119,0.917720,83.369167,RELIABLE
3,DST_0004,0.453431,0.438038,0.339220,41.783333,UNKNOWN
4,DST_0005,0.498039,0.576765,0.265248,15.612500,UNKNOWN
5,DST_0006,0.513725,0.465464,0.301605,55.080833,UNKNOWN
6,DST_0007,0.515196,0.930945,0.406261,72.070000,RELIABLE
7,DST_0008,0.873529,0.926553,0.365961,57.322500,UNKNOWN
8,DST_0009,0.811765,0.672621,0.493533,68.176250,RELIABLE
9,DST_0010,0.284314,0.452912,0.268485,36.100000,UNKNOWN


In [63]:
# === PHASE 4: MASTER COMPONENT VALIDATION ===

required_columns = [
    "district_id",
    "Demand_Signal",
    "Mean_Distribution_Gap",
    "Competitive_Intensity",
    "Mean_Panel_Coverage_Pct",
    "Coverage_Status"
]

missing_columns = [
    col for col in required_columns
    if col not in phase4_final.columns
]

print("=== PHASE 4 MASTER COMPONENT VALIDATION ===")
print("Districts:", len(phase4_final))
print("Missing required columns:", missing_columns)

for col in required_columns[1:]:
    print(
        f"Missing {col}:",
        phase4_final[col].isna().sum()
    )

print(
    "\nDuplicate district IDs:",
    phase4_final["district_id"].duplicated().sum()
)

if (
    len(phase4_final) == 340
    and len(missing_columns) == 0
    and phase4_final[required_columns[1:]].isna().sum().sum() == 0
    and phase4_final["district_id"].duplicated().sum() == 0
):
    print("\n✓ PHASE 4 MASTER VALIDATION PASSED")
else:
    print("\n⚠ PHASE 4 MASTER VALIDATION NEEDS REVIEW")

=== PHASE 4 MASTER COMPONENT VALIDATION ===
Districts: 340
Missing required columns: []
Missing Demand_Signal: 0
Missing Mean_Distribution_Gap: 0
Missing Competitive_Intensity: 0
Missing Mean_Panel_Coverage_Pct: 0
Missing Coverage_Status: 0

Duplicate district IDs: 0

✓ PHASE 4 MASTER VALIDATION PASSED


In [74]:
# === WHITESPACE CONFIDENCE INDEX ===

phase4_final["WCI"] = (
    0.45 * phase4_final["Demand_Signal"]
    + 0.35 * phase4_final["Mean_Distribution_Gap"]
    + 0.20 * (1 - phase4_final["Competitive_Intensity"])
)

print("=== WHITESPACE CONFIDENCE INDEX ===")
print("Districts:", len(phase4_final))
print("Missing WCI:", phase4_final["WCI"].isna().sum())
print(
    "WCI range:",
    round(phase4_final["WCI"].min(), 4),
    "to",
    round(phase4_final["WCI"].max(), 4)
)

print("\n=== TOP WCI DISTRICTS ===")

top_wci = (
    phase4_final[
        [
            "district_id",
            "Demand_Signal",
            "Mean_Distribution_Gap",
            "Competitive_Intensity",
            "Mean_Panel_Coverage_Pct",
            "Coverage_Status",
            "WCI"
        ]
    ]
    .sort_values("WCI", ascending=False)
    .reset_index(drop=True)
)

display(top_wci.head(20))

=== WHITESPACE CONFIDENCE INDEX ===
Districts: 340
Missing WCI: 0
WCI range: 0.237 to 0.8762

=== TOP WCI DISTRICTS ===


,district_id,Demand_Signal,Mean_Distribution_Gap,Competitive_Intensity,Mean_Panel_Coverage_Pct,Coverage_Status,WCI
0,DST_0040,0.935294,0.931602,0.353842,66.228750,RELIABLE,0.876175
1,DST_0192,0.944608,0.924601,0.377150,60.542917,RELIABLE,0.873254
2,DST_0211,0.919608,0.928409,0.334700,60.326667,RELIABLE,0.871827
3,DST_0301,0.908333,0.933118,0.354095,73.367083,RELIABLE,0.864523
4,DST_0271,0.914216,0.930131,0.373373,70.215417,RELIABLE,0.862268
5,DST_0068,0.900000,0.923955,0.371566,55.510000,UNKNOWN,0.854071
6,DST_0069,0.888725,0.922267,0.379218,62.027083,RELIABLE,0.846876
7,DST_0008,0.873529,0.926553,0.365961,57.322500,UNKNOWN,0.844190
8,DST_0244,0.861275,0.927974,0.358256,73.799167,RELIABLE,0.840713
9,DST_0329,0.865686,0.928676,0.383180,71.464583,RELIABLE,0.837959


In [76]:
# === RECOMMENDED TARGET CLASSIFICATION ===

phase4_final["Recommended_Target"] = np.where(
    (
        (phase4_final["WCI"] >= 0.70)
        & (phase4_final["Mean_Distribution_Gap"] >= 0.40)
        & (phase4_final["Mean_Panel_Coverage_Pct"] >= 60)
    ),
    "RECOMMENDED",
    "NOT_RECOMMENDED"
)

print("=== RECOMMENDED TARGET DISTRIBUTION ===")
print(
    phase4_final["Recommended_Target"]
    .value_counts()
)

recommended_targets = (
    phase4_final[
        phase4_final["Recommended_Target"] == "RECOMMENDED"
    ]
    .sort_values("WCI", ascending=False)
    .reset_index(drop=True)
)

print("\n=== RECOMMENDED TARGETS ===")
print("Recommended districts:", len(recommended_targets))

display(
    recommended_targets[
        [
            "district_id",
            "Demand_Signal",
            "Mean_Distribution_Gap",
            "Competitive_Intensity",
            "Mean_Panel_Coverage_Pct",
            "Coverage_Status",
            "WCI",
            "WCI_Priority",
            "Recommended_Target"
        ]
    ]
)

=== RECOMMENDED TARGET DISTRIBUTION ===
Recommended_Target
NOT_RECOMMENDED    315
RECOMMENDED         25
Name: count, dtype: int64

=== RECOMMENDED TARGETS ===
Recommended districts: 25


,district_id,Demand_Signal,Mean_Distribution_Gap,Competitive_Intensity,Mean_Panel_Coverage_Pct,Coverage_Status,WCI,WCI_Priority,Recommended_Target
0,DST_0040,0.935294,0.931602,0.353842,66.228750,RELIABLE,0.876175,VERY_HIGH,RECOMMENDED
1,DST_0192,0.944608,0.924601,0.377150,60.542917,RELIABLE,0.873254,VERY_HIGH,RECOMMENDED
2,DST_0211,0.919608,0.928409,0.334700,60.326667,RELIABLE,0.871827,VERY_HIGH,RECOMMENDED
3,DST_0301,0.908333,0.933118,0.354095,73.367083,RELIABLE,0.864523,VERY_HIGH,RECOMMENDED
4,DST_0271,0.914216,0.930131,0.373373,70.215417,RELIABLE,0.862268,VERY_HIGH,RECOMMENDED
5,DST_0069,0.888725,0.922267,0.379218,62.027083,RELIABLE,0.846876,HIGH,RECOMMENDED
6,DST_0244,0.861275,0.927974,0.358256,73.799167,RELIABLE,0.840713,HIGH,RECOMMENDED
7,DST_0329,0.865686,0.928676,0.383180,71.464583,RELIABLE,0.837959,HIGH,RECOMMENDED
8,DST_0246,0.852941,0.929079,0.362217,69.429583,RELIABLE,0.836558,HIGH,RECOMMENDED
9,DST_0154,0.866667,0.926849,0.390137,65.090833,RELIABLE,0.836370,HIGH,RECOMMENDED


In [77]:
# === RECOMMENDED TARGETS WITH DISTRICT NAMES ===

recommended_targets = recommended_targets.merge(
    district_master[
        ["District_Code", "District_Name", "State_Union_Territory"]
    ],
    left_on="district_id",
    right_on="District_Code",
    how="left",
    validate="one_to_one"
)

recommended_targets = (
    recommended_targets[
        [
            "district_id",
            "District_Name",
            "State_Union_Territory",
            "Demand_Signal",
            "Mean_Distribution_Gap",
            "Competitive_Intensity",
            "Mean_Panel_Coverage_Pct",
            "Coverage_Status",
            "WCI",
            "WCI_Priority",
            "Recommended_Target"
        ]
    ]
    .sort_values("WCI", ascending=False)
    .reset_index(drop=True)
)

print("=== RECOMMENDED TARGETS WITH DISTRICT NAMES ===")
print("Recommended districts:", len(recommended_targets))
print("Missing district names:",
      recommended_targets["District_Name"].isna().sum())

display(recommended_targets)

=== RECOMMENDED TARGETS WITH DISTRICT NAMES ===
Recommended districts: 25
Missing district names: 0


,district_id,District_Name,State_Union_Territory,Demand_Signal,Mean_Distribution_Gap,Competitive_Intensity,Mean_Panel_Coverage_Pct,Coverage_Status,WCI,WCI_Priority,Recommended_Target
0,DST_0040,Dharwad,Karnataka,0.935294,0.931602,0.353842,66.228750,RELIABLE,0.876175,VERY_HIGH,RECOMMENDED
1,DST_0192,Paschim Bardhaman,West Bengal,0.944608,0.924601,0.377150,60.542917,RELIABLE,0.873254,VERY_HIGH,RECOMMENDED
2,DST_0211,Warangal,Telangana,0.919608,0.928409,0.334700,60.326667,RELIABLE,0.871827,VERY_HIGH,RECOMMENDED
3,DST_0301,Ujjain,Madhya Pradesh,0.908333,0.933118,0.354095,73.367083,RELIABLE,0.864523,VERY_HIGH,RECOMMENDED
4,DST_0271,Vizag,Andhra Pradesh,0.914216,0.930131,0.373373,70.215417,RELIABLE,0.862268,VERY_HIGH,RECOMMENDED
5,DST_0069,Salem,Tamil Nadu,0.888725,0.922267,0.379218,62.027083,RELIABLE,0.846876,HIGH,RECOMMENDED
6,DST_0244,Udaipur,Rajasthan,0.861275,0.927974,0.358256,73.799167,RELIABLE,0.840713,HIGH,RECOMMENDED
7,DST_0329,Kozhikode,Kerala,0.865686,0.928676,0.383180,71.464583,RELIABLE,0.837959,HIGH,RECOMMENDED
8,DST_0246,Alwar,Rajasthan,0.852941,0.929079,0.362217,69.429583,RELIABLE,0.836558,HIGH,RECOMMENDED
9,DST_0154,Rajkot,Gujarat,0.866667,0.926849,0.390137,65.090833,RELIABLE,0.836370,HIGH,RECOMMENDED


In [78]:
# === RECOMMENDED TARGET VALIDATION ===

validation_errors = []

# 1. All recommended targets must satisfy the three assessment conditions
invalid_targets = recommended_targets[
    ~(
        (recommended_targets["WCI"] >= 0.70)
        & (recommended_targets["Mean_Distribution_Gap"] >= 0.40)
        & (recommended_targets["Mean_Panel_Coverage_Pct"] >= 60)
    )
]

# 2. No UNKNOWN coverage district should be recommended
unknown_recommended = recommended_targets[
    recommended_targets["Coverage_Status"] == "UNKNOWN"
]

# 3. District IDs must be unique
duplicate_ids = recommended_targets["district_id"].duplicated().sum()

# 4. District names must be present
missing_names = recommended_targets["District_Name"].isna().sum()

print("=== RECOMMENDED TARGET VALIDATION ===")
print("Recommended districts:", len(recommended_targets))
print("Invalid recommended targets:", len(invalid_targets))
print("UNKNOWN districts recommended:", len(unknown_recommended))
print("Duplicate district IDs:", duplicate_ids)
print("Missing district names:", missing_names)

if (
    len(invalid_targets) == 0
    and len(unknown_recommended) == 0
    and duplicate_ids == 0
    and missing_names == 0
):
    print("\n✓ RECOMMENDED TARGET VALIDATION PASSED")
else:
    print("\n⚠ RECOMMENDED TARGET VALIDATION FAILED")

=== RECOMMENDED TARGET VALIDATION ===
Recommended districts: 25
Invalid recommended targets: 0
UNKNOWN districts recommended: 0
Duplicate district IDs: 0
Missing district names: 0

✓ RECOMMENDED TARGET VALIDATION PASSED


In [79]:
# === FINAL WCI OUTPUTS SAVED ===

OUTPUT_DATA.mkdir(parents=True, exist_ok=True)

# 1. Complete WCI analysis for all 340 districts
phase4_final.to_csv(
    OUTPUT_DATA / "wci_district_analysis.csv",
    index=False
)

# 2. Ranked WCI table
wci_ranked = (
    phase4_final
    .sort_values("WCI", ascending=False)
    .reset_index(drop=True)
)

wci_ranked.insert(
    0,
    "WCI_Rank",
    range(1, len(wci_ranked) + 1)
)

wci_ranked.to_csv(
    OUTPUT_DATA / "wci_ranked_districts.csv",
    index=False
)

# 3. Recommended expansion targets
recommended_targets.to_csv(
    OUTPUT_DATA / "recommended_targets.csv",
    index=False
)

# 4. UNKNOWN districts kept separately
unknown_districts_final = (
    phase4_final[
        phase4_final["Coverage_Status"] == "UNKNOWN"
    ]
    .sort_values("WCI", ascending=False)
    .reset_index(drop=True)
)

unknown_districts_final.to_csv(
    OUTPUT_DATA / "unknown_districts_final.csv",
    index=False
)

# 5. HIGH / VERY_HIGH WCI districts
high_priority_wci = (
    wci_ranked[
        wci_ranked["WCI_Priority"].isin(
            ["HIGH", "VERY_HIGH"]
        )
    ]
    .reset_index(drop=True)
)

high_priority_wci.to_csv(
    OUTPUT_DATA / "high_priority_wci_districts.csv",
    index=False
)

print("=== FINAL WCI OUTPUTS SAVED ===")
print("✓ wci_district_analysis.csv")
print("✓ wci_ranked_districts.csv")
print("✓ recommended_targets.csv")
print("✓ unknown_districts_final.csv")
print("✓ high_priority_wci_districts.csv")

print("\nRows saved:")
print("Complete WCI table:", len(phase4_final))
print("Ranked WCI table:", len(wci_ranked))
print("Recommended targets:", len(recommended_targets))
print("UNKNOWN districts:", len(unknown_districts_final))
print("High-priority districts:", len(high_priority_wci))

=== FINAL WCI OUTPUTS SAVED ===
✓ wci_district_analysis.csv
✓ wci_ranked_districts.csv
✓ recommended_targets.csv
✓ unknown_districts_final.csv
✓ high_priority_wci_districts.csv

Rows saved:
Complete WCI table: 340
Ranked WCI table: 340
Recommended targets: 25
UNKNOWN districts: 257
High-priority districts: 28


In [82]:
# === FINAL DECISION TABLE ===

# Add district names and states from District Master
wci_ranked_named = wci_ranked.merge(
    district_master[
        [
            "District_Code",
            "District_Name",
            "State_Union_Territory"
        ]
    ],
    left_on="district_id",
    right_on="District_Code",
    how="left",
    validate="one_to_one"
)

# Remove duplicate join key
wci_ranked_named = wci_ranked_named.drop(
    columns=["District_Code"]
)

# Create final decision table
final_decision_table = (
    wci_ranked_named[
        [
            "WCI_Rank",
            "district_id",
            "District_Name",
            "State_Union_Territory",
            "Demand_Signal",
            "Mean_Distribution_Gap",
            "Competitive_Intensity",
            "Mean_Panel_Coverage_Pct",
            "Coverage_Status",
            "WCI",
            "WCI_Priority"
        ]
    ]
    .copy()
)

print("=== FINAL DECISION TABLE ===")
print("Districts:", len(final_decision_table))
print(
    "Unique districts:",
    final_decision_table["district_id"].nunique()
)
print(
    "Missing district names:",
    final_decision_table["District_Name"].isna().sum()
)
print(
    "Missing states:",
    final_decision_table["State_Union_Territory"].isna().sum()
)

display(final_decision_table.head(30))

=== FINAL DECISION TABLE ===
Districts: 340
Unique districts: 340
Missing district names: 0
Missing states: 0


,WCI_Rank,district_id,District_Name,State_Union_Territory,Demand_Signal,Mean_Distribution_Gap,Competitive_Intensity,Mean_Panel_Coverage_Pct,Coverage_Status,WCI,WCI_Priority
0,1,DST_0040,Dharwad,Karnataka,0.935294,0.931602,0.353842,66.228750,RELIABLE,0.876175,VERY_HIGH
1,2,DST_0192,Paschim Bardhaman,West Bengal,0.944608,0.924601,0.377150,60.542917,RELIABLE,0.873254,VERY_HIGH
2,3,DST_0211,Warangal,Telangana,0.919608,0.928409,0.334700,60.326667,RELIABLE,0.871827,VERY_HIGH
3,4,DST_0301,Ujjain,Madhya Pradesh,0.908333,0.933118,0.354095,73.367083,RELIABLE,0.864523,VERY_HIGH
4,5,DST_0271,Vizag,Andhra Pradesh,0.914216,0.930131,0.373373,70.215417,RELIABLE,0.862268,VERY_HIGH
5,6,DST_0068,Tiruchirappalli,Tamil Nadu,0.900000,0.923955,0.371566,55.510000,UNKNOWN,0.854071,VERY_HIGH
6,7,DST_0069,Salem,Tamil Nadu,0.888725,0.922267,0.379218,62.027083,RELIABLE,0.846876,HIGH
7,8,DST_0008,Aurangabad,Maharashtra,0.873529,0.926553,0.365961,57.322500,UNKNOWN,0.844190,HIGH
8,9,DST_0244,Udaipur,Rajasthan,0.861275,0.927974,0.358256,73.799167,RELIABLE,0.840713,HIGH
9,10,DST_0329,Kozhikode,Kerala,0.865686,0.928676,0.383180,71.464583,RELIABLE,0.837959,HIGH


In [84]:
# === FINAL WCI OUTPUTS SAVED ===

OUTPUT_DATA.mkdir(parents=True, exist_ok=True)

# Complete WCI analysis
wci_ranked_named.to_csv(
    OUTPUT_DATA / "wci_complete_analysis.csv",
    index=False
)

# Final decision table
final_decision_table.to_csv(
    OUTPUT_DATA / "final_wci_decision_table.csv",
    index=False
)

# UNKNOWN / low-coverage districts
unknown_districts_final = (
    wci_ranked_named[
        wci_ranked_named["Coverage_Status"] == "UNKNOWN"
    ]
    .copy()
)

unknown_districts_final.to_csv(
    OUTPUT_DATA / "unknown_districts_final.csv",
    index=False
)

# HIGH + VERY_HIGH priority districts
high_priority_wci = (
    wci_ranked_named[
        wci_ranked_named["WCI_Priority"].isin(
            ["HIGH", "VERY_HIGH"]
        )
    ]
    .copy()
)

high_priority_wci.to_csv(
    OUTPUT_DATA / "high_priority_wci_districts.csv",
    index=False
)

print("=== FINAL WCI OUTPUTS SAVED ===")
print("✓ final_wci_decision_table.csv")
print("✓ wci_complete_analysis.csv")
print("✓ unknown_districts_final.csv")
print("✓ high_priority_wci_districts.csv")

print("\nRows saved:")
print("Final decision table:", len(final_decision_table))
print("Complete WCI analysis:", len(wci_ranked_named))
print("UNKNOWN districts:", len(unknown_districts_final))
print("High-priority districts:", len(high_priority_wci))

=== FINAL WCI OUTPUTS SAVED ===
✓ final_wci_decision_table.csv
✓ wci_complete_analysis.csv
✓ unknown_districts_final.csv
✓ high_priority_wci_districts.csv

Rows saved:
Final decision table: 340
Complete WCI analysis: 340
UNKNOWN districts: 257
High-priority districts: 28


In [86]:
# === FINAL OVERALL VALIDATION ===

required_columns = [
    "WCI_Rank",
    "district_id",
    "District_Name",
    "State_Union_Territory",
    "Demand_Signal",
    "Mean_Distribution_Gap",
    "Competitive_Intensity",
    "Mean_Panel_Coverage_Pct",
    "Coverage_Status",
    "WCI",
    "WCI_Priority"
]

missing_columns = [
    col for col in required_columns
    if col not in wci_ranked_named.columns
]

print("=== FINAL OVERALL VALIDATION ===")
print("Total districts:", len(wci_ranked_named))
print(
    "Unique district IDs:",
    wci_ranked_named["district_id"].nunique()
)
print("Missing required columns:", missing_columns)

print("\nMissing values:")
print(
    wci_ranked_named[required_columns]
    .isna()
    .sum()
)

print(
    "\nDuplicate district IDs:",
    wci_ranked_named["district_id"].duplicated().sum()
)

print("\nWCI bounds:")
print("Minimum:", round(wci_ranked_named["WCI"].min(), 4))
print("Maximum:", round(wci_ranked_named["WCI"].max(), 4))

print("\nPriority distribution:")
print(
    wci_ranked_named["WCI_Priority"].value_counts()
)

if (
    len(wci_ranked_named) == 340
    and wci_ranked_named["district_id"].nunique() == 340
    and len(missing_columns) == 0
    and wci_ranked_named[required_columns].isna().sum().sum() == 0
    and wci_ranked_named["district_id"].duplicated().sum() == 0
    and wci_ranked_named["WCI"].between(0, 1).all()
    and wci_ranked_named["District_Name"].nunique() == 340
):
    print("\n✓ FINAL OVERALL VALIDATION PASSED")
else:
    print("\n⚠ FINAL OVERALL VALIDATION FAILED")

=== FINAL OVERALL VALIDATION ===
Total districts: 340
Unique district IDs: 340
Missing required columns: []

Missing values:
WCI_Rank                   0
district_id                0
District_Name              0
State_Union_Territory      0
Demand_Signal              0
Mean_Distribution_Gap      0
Competitive_Intensity      0
Mean_Panel_Coverage_Pct    0
Coverage_Status            0
WCI                        0
WCI_Priority               0
dtype: int64

Duplicate district IDs: 0

WCI bounds:
Minimum: 0.237
Maximum: 0.8762

Priority distribution:
WCI_Priority
MEDIUM       200
LOW          112
HIGH          22
VERY_HIGH      6
Name: count, dtype: int64

✓ FINAL OVERALL VALIDATION PASSED


In [87]:
# === DEMAND SIGNAL: MARKET-SIZE NORMALIZATION CHECK ===

print("=== DEMAND SIGNAL COMPONENT CHECK ===")

demand_columns = [
    col for col in phase4_final.columns
    if "demand" in col.lower()
    or "velocity" in col.lower()
    or "audience" in col.lower()
    or "engagement" in col.lower()
    or "event" in col.lower()
]

print("Demand-related columns found:")
for col in demand_columns:
    print("✓", col)

print("\nDistrict-level Demand Signal:")
print("Minimum:", round(phase4_final["Demand_Signal"].min(), 4))
print("Maximum:", round(phase4_final["Demand_Signal"].max(), 4))
print("Mean:", round(phase4_final["Demand_Signal"].mean(), 4))
print("Unique districts:", phase4_final["district_id"].nunique())

print("\nMarket-size columns available:")
market_columns = [
    col for col in phase4_final.columns
    if any(x in col.lower() for x in [
        "population",
        "retail_universe",
        "market_size",
        "universe"
    ])
]

for col in market_columns:
    print("✓", col)

display(
    phase4_final[
        ["district_id", "Demand_Signal"] + market_columns
    ]
    .sort_values("Demand_Signal", ascending=False)
    .head(20)
)

=== DEMAND SIGNAL COMPONENT CHECK ===
Demand-related columns found:
✓ Demand_Signal

District-level Demand Signal:
Minimum: 0.0088
Maximum: 0.9686
Mean: 0.5015
Unique districts: 340

Market-size columns available:
✓ Total_Population
✓ Estimated_Total_Retail_Universe


,district_id,Demand_Signal,Total_Population,Estimated_Total_Retail_Universe
240,DST_0241,0.968627,1951014,3199
271,DST_0272,0.958333,2218000,3948
99,DST_0100,0.956863,582320,1281
272,DST_0273,0.954902,2091000,3575
212,DST_0213,0.952451,1005711,1241
191,DST_0192,0.944608,2882031,5533
39,DST_0040,0.935294,1847023,2951
242,DST_0243,0.933333,2583052,3512
329,DST_0330,0.930392,3121200,5424
296,DST_0297,0.929902,3276697,6016


In [88]:
# === MARKET SIZE NORMALIZATION CHECK ===

print("=== MARKET SIZE NORMALIZATION INPUTS ===")

required_market_columns = [
    "district_id",
    "Total_Population",
    "Estimated_Total_Retail_Universe",
    "Demand_Signal"
]

missing_market_columns = [
    col for col in required_market_columns
    if col not in phase4_final.columns
]

print("Missing required columns:", missing_market_columns)

if len(missing_market_columns) == 0:
    print("\nMarket-size inputs available:")
    print("✓ Total_Population")
    print("✓ Estimated_Total_Retail_Universe")
    print("✓ Demand_Signal")

    print("\nPopulation range:")
    print(
        phase4_final["Total_Population"].min(),
        "to",
        phase4_final["Total_Population"].max()
    )

    print("\nRetail-universe range:")
    print(
        phase4_final["Estimated_Total_Retail_Universe"].min(),
        "to",
        phase4_final["Estimated_Total_Retail_Universe"].max()
    )

    print("\nCurrent Demand Signal range:")
    print(
        round(phase4_final["Demand_Signal"].min(), 4),
        "to",
        round(phase4_final["Demand_Signal"].max(), 4)
    )

    display(
        phase4_final[
            [
                "district_id",
                "Total_Population",
                "Estimated_Total_Retail_Universe",
                "Demand_Signal"
            ]
        ]
        .sort_values("Demand_Signal", ascending=False)
        .head(20)
    )
else:
    print("\n⚠ Market-size columns are missing.")

=== MARKET SIZE NORMALIZATION INPUTS ===
Missing required columns: []

Market-size inputs available:
✓ Total_Population
✓ Estimated_Total_Retail_Universe
✓ Demand_Signal

Population range:
142004 to 10009781

Retail-universe range:
217 to 18500

Current Demand Signal range:
0.0088 to 0.9686


,district_id,Total_Population,Estimated_Total_Retail_Universe,Demand_Signal
240,DST_0241,1951014,3199,0.968627
271,DST_0272,2218000,3948,0.958333
99,DST_0100,582320,1281,0.956863
272,DST_0273,2091000,3575,0.954902
212,DST_0213,1005711,1241,0.952451
191,DST_0192,2882031,5533,0.944608
39,DST_0040,1847023,2951,0.935294
242,DST_0243,2583052,3512,0.933333
329,DST_0330,3121200,5424,0.930392
296,DST_0297,3276697,6016,0.929902


In [89]:
# === MARKET-SIZE-NORMALIZED DEMAND SIGNAL ===

# Normalize estimated retail universe across districts
retail_universe_min = phase4_final["Estimated_Total_Retail_Universe"].min()
retail_universe_max = phase4_final["Estimated_Total_Retail_Universe"].max()

phase4_final["Market_Size_Score"] = (
    (phase4_final["Estimated_Total_Retail_Universe"] - retail_universe_min)
    / (retail_universe_max - retail_universe_min)
)

# Adjust demand signal for market size
phase4_final["Market_Size_Adjusted_Demand"] = (
    phase4_final["Demand_Signal"]
    * phase4_final["Market_Size_Score"]
)

print("=== MARKET-SIZE-ADJUSTED DEMAND ===")
print("Districts:", len(phase4_final))
print(
    "Market Size Score range:",
    round(phase4_final["Market_Size_Score"].min(), 4),
    "to",
    round(phase4_final["Market_Size_Score"].max(), 4)
)
print(
    "Adjusted Demand range:",
    round(phase4_final["Market_Size_Adjusted_Demand"].min(), 4),
    "to",
    round(phase4_final["Market_Size_Adjusted_Demand"].max(), 4)
)

display(
    phase4_final[
        [
            "district_id",
            "Demand_Signal",
            "Estimated_Total_Retail_Universe",
            "Market_Size_Score",
            "Market_Size_Adjusted_Demand"
        ]
    ]
    .sort_values("Market_Size_Adjusted_Demand", ascending=False)
    .head(20)
)

=== MARKET-SIZE-ADJUSTED DEMAND ===
Districts: 340
Market Size Score range: 0.0 to 1.0
Adjusted Demand range: 0.0 to 0.5331


,district_id,Demand_Signal,Estimated_Total_Retail_Universe,Market_Size_Score,Market_Size_Adjusted_Demand
2,DST_0003,0.633824,15596,0.841164,0.533149
35,DST_0036,0.497059,18500,1.000000,0.497059
0,DST_0001,0.495098,18500,1.000000,0.495098
150,DST_0151,0.631373,14255,0.767817,0.484779
184,DST_0185,0.505392,15995,0.862987,0.436147
64,DST_0065,0.512255,15594,0.841055,0.430834
206,DST_0207,0.852451,8675,0.462616,0.394357
183,DST_0184,0.714216,9892,0.529180,0.377949
3,DST_0004,0.453431,15155,0.817043,0.370473
151,DST_0152,0.546569,11676,0.626757,0.342566


In [92]:
# === TOP 5 OPPORTUNITY SIZING ===

kestrel = pd.read_csv(
    PROCESSED_DATA / "kestrel_processed.csv"
)

print("=== KESTREL SALES LOADED ===")
print("Rows:", len(kestrel))
print("Columns:", kestrel.columns.tolist())

display(kestrel.head())

=== KESTREL SALES LOADED ===
Rows: 48960
Columns: ['dispatch_date', 'month_id', 'district_id', 'district_name', 'state', 'sku_id', 'sku_name', 'factory_dispatch_units', 'distributor_sellout_units', 'reported_stocking_outlets', 'net_billing_revenue_inr', 'distributor_code', 'data_source', 'dispatch_date_parsed', 'dispatch_date_is_future', 'district_id_original', 'district_id_normalized', 'district_name_standardized']


,dispatch_date,month_id,district_id,district_name,state,sku_id,sku_name,factory_dispatch_units,distributor_sellout_units,reported_stocking_outlets,net_billing_revenue_inr,distributor_code,data_source,dispatch_date_parsed,dispatch_date_is_future,district_id_original,district_id_normalized,district_name_standardized
0,10-07-25,2025-10,20,Chandrapur,Maharashtra,KES_TROP_250,Kestrel Tropical Citrus 250ml,3882,3409,896,294537.6,DST-IN-020,ERP_SAP_ECC6_FEED,2025-10-07 00:00:00+00:00,False,20,DST_0020,Chandrapur
1,22-02-2026,2026-02,DST_0103,North Delhi,Delhi NCR,KES_ORIG_330,Kestrel Sleek Can 330ml,2608,2354,642,228808.8,DST-IN-103,ERP_SAP_ECC6_FEED,2026-02-22 00:00:00+00:00,False,DST_0103,DST_0103,North Delhi
2,07-14-25,2025-07,DST_0193,Nadia,West Bengal,KES_PACK4_250,Kestrel Multipack 4x250ml,8452,7695,1630,2326968.0,DST-IN-193,ERP_SAP_ECC6_FEED,2025-07-14 00:00:00+00:00,False,DST_0193,DST_0193,Nadia
3,12-06-2026,2026-06,21,Yavatmal,Maharashtra,KES_ZERO_250,Kestrel Zero Sugar 250ml,5985,6296,1215,521308.8,DST-IN-021,ERP_SAP_ECC6_FEED,NaN,True,21,DST_0021,Yavatmal
4,12-05-2026,2026-05,DST_0268,Jaisalmer,Rajasthan,KES_ZERO_250,Kestrel Zero Sugar 250ml,112,111,91,9190.8,DST-IN-268,ERP_SAP_ECC6_FEED,NaN,True,DST_0268,DST_0268,Jaisalmer


In [93]:
# === TOP 5 WCI OPPORTUNITY SIZING ===

top5_targets = (
    wci_ranked
    .head(5)
    .copy()
)

print("=== TOP 5 WCI TARGETS ===")
display(
    top5_targets[
        [
            "WCI_Rank",
            "district_id",
            "Demand_Signal",
            "Mean_Distribution_Gap",
            "Competitive_Intensity",
            "WCI",
            "WCI_Priority"
        ]
    ]
)

=== TOP 5 WCI TARGETS ===


,WCI_Rank,district_id,Demand_Signal,Mean_Distribution_Gap,Competitive_Intensity,WCI,WCI_Priority
0,1,DST_0040,0.935294,0.931602,0.353842,0.876175,VERY_HIGH
1,2,DST_0192,0.944608,0.924601,0.377150,0.873254,VERY_HIGH
2,3,DST_0211,0.919608,0.928409,0.334700,0.871827,VERY_HIGH
3,4,DST_0301,0.908333,0.933118,0.354095,0.864523,VERY_HIGH
4,5,DST_0271,0.914216,0.930131,0.373373,0.862268,VERY_HIGH


In [94]:
# === TOP 5 ₹ OPPORTUNITY SIZING ===

kestrel_top5 = (
    kestrel[
        kestrel["district_id"].astype(str).isin(
            top5_targets["district_id"].astype(str)
        )
    ]
    .copy()
)

# Total historical Kestrel revenue by district
top5_revenue = (
    kestrel_top5
    .groupby("district_id", as_index=False)
    .agg(
        Historical_Revenue_INR=("net_billing_revenue_inr", "sum")
    )
)

# Convert 24-month historical revenue to annual run-rate
top5_revenue["Annual_Revenue_INR"] = (
    top5_revenue["Historical_Revenue_INR"] / 2
)

# Attach WCI metrics
top5_sized = (
    top5_targets
    .merge(
        top5_revenue,
        on="district_id",
        how="left"
    )
)

print("=== TOP 5 ₹ OPPORTUNITY SIZING ===")
print("Top 5 districts:", len(top5_sized))
print("Missing revenue:", top5_sized["Historical_Revenue_INR"].isna().sum())

display(
    top5_sized[
        [
            "WCI_Rank",
            "district_id",
            "Demand_Signal",
            "Mean_Distribution_Gap",
            "Competitive_Intensity",
            "WCI",
            "WCI_Priority",
            "Historical_Revenue_INR",
            "Annual_Revenue_INR"
        ]
    ]
)

=== TOP 5 ₹ OPPORTUNITY SIZING ===
Top 5 districts: 5
Missing revenue: 0


,WCI_Rank,district_id,Demand_Signal,Mean_Distribution_Gap,Competitive_Intensity,WCI,WCI_Priority,Historical_Revenue_INR,Annual_Revenue_INR
0,1,DST_0040,0.935294,0.931602,0.353842,0.876175,VERY_HIGH,19378512.0,9689256.0
1,2,DST_0192,0.944608,0.924601,0.377150,0.873254,VERY_HIGH,29683155.6,14841577.8
2,3,DST_0211,0.919608,0.928409,0.334700,0.871827,VERY_HIGH,6293494.8,3146747.4
3,4,DST_0301,0.908333,0.933118,0.354095,0.864523,VERY_HIGH,19450339.2,9725169.6
4,5,DST_0271,0.914216,0.930131,0.373373,0.862268,VERY_HIGH,29410632.0,14705316.0


In [95]:
# === TOP 5 TARGET RECOMMENDATIONS ===

top5_sized = top5_sized.merge(
    district_master[
        [
            "District_Code",
            "District_Name",
            "State_Union_Territory"
        ]
    ],
    left_on="district_id",
    right_on="District_Code",
    how="left",
    validate="one_to_one"
)

# Intervention logic
top5_sized["Recommended_Intervention"] = np.select(
    [
        (top5_sized["Demand_Signal"] >= 0.70) &
        (top5_sized["Mean_Distribution_Gap"] >= 0.40),

        (top5_sized["Mean_Distribution_Gap"] >= 0.40),

        (top5_sized["Demand_Signal"] >= 0.70)
    ],
    [
        "BOTH",
        "DISTRIBUTION",
        "MARKETING"
    ],
    default="REVIEW"
)

print("=== TOP 5 TARGETS WITH INTERVENTION ===")

display(
    top5_sized[
        [
            "WCI_Rank",
            "district_id",
            "District_Name",
            "State_Union_Territory",
            "Demand_Signal",
            "Mean_Distribution_Gap",
            "Competitive_Intensity",
            "WCI",
            "Historical_Revenue_INR",
            "Annual_Revenue_INR",
            "Recommended_Intervention"
        ]
    ]
)

=== TOP 5 TARGETS WITH INTERVENTION ===


,WCI_Rank,district_id,District_Name,State_Union_Territory,Demand_Signal,Mean_Distribution_Gap,Competitive_Intensity,WCI,Historical_Revenue_INR,Annual_Revenue_INR,Recommended_Intervention
0,1,DST_0040,Dharwad,Karnataka,0.935294,0.931602,0.353842,0.876175,19378512.0,9689256.0,BOTH
1,2,DST_0192,Paschim Bardhaman,West Bengal,0.944608,0.924601,0.377150,0.873254,29683155.6,14841577.8,BOTH
2,3,DST_0211,Warangal,Telangana,0.919608,0.928409,0.334700,0.871827,6293494.8,3146747.4,BOTH
3,4,DST_0301,Ujjain,Madhya Pradesh,0.908333,0.933118,0.354095,0.864523,19450339.2,9725169.6,BOTH
4,5,DST_0271,Vizag,Andhra Pradesh,0.914216,0.930131,0.373373,0.862268,29410632.0,14705316.0,BOTH


In [107]:
# === SAVE FINAL TOP 5 OPPORTUNITY SIZING ===

OUTPUT_DATA.mkdir(parents=True, exist_ok=True)

top5_sized.to_csv(
    OUTPUT_DATA / "top5_revenue_opportunity.csv",
    index=False
)

print("=== TOP 5 OPPORTUNITY OUTPUT SAVED ===")
print(
    "File:",
    OUTPUT_DATA / "top5_revenue_opportunity.csv"
)
print(
    "Rows:",
    len(top5_sized)
)

=== TOP 5 OPPORTUNITY OUTPUT SAVED ===
File: d:\sivahitesh\Desktop\Tail_project\data\outputs\top5_revenue_opportunity.csv
Rows: 5


In [96]:
# === PALE DISTRICT REJECTION SETUP ===

# Calculate annualized Kestrel revenue for all districts
district_revenue = (
    kestrel
    .groupby("district_id", as_index=False)
    .agg(
        Historical_Revenue_INR=("net_billing_revenue_inr", "sum")
    )
)

district_revenue["Annual_Revenue_INR"] = (
    district_revenue["Historical_Revenue_INR"] / 2
)

# Attach revenue to the complete WCI table
pale_analysis = (
    wci_ranked
    .merge(
        district_revenue[
            [
                "district_id",
                "Historical_Revenue_INR",
                "Annual_Revenue_INR"
            ]
        ],
        on="district_id",
        how="left"
    )
)

print("=== PALE DISTRICT REJECTION SETUP ===")
print("Districts:", len(pale_analysis))
print("Missing revenue:", pale_analysis["Annual_Revenue_INR"].isna().sum())

display(
    pale_analysis[
        [
            "district_id",
            "Demand_Signal",
            "Mean_Distribution_Gap",
            "Competitive_Intensity",
            "Mean_Panel_Coverage_Pct",
            "Coverage_Status",
            "WCI",
            "WCI_Priority",
            "Annual_Revenue_INR"
        ]
    ].sort_values("Annual_Revenue_INR").head(20)
)

=== PALE DISTRICT REJECTION SETUP ===
Districts: 340
Missing revenue: 0


,district_id,Demand_Signal,Mean_Distribution_Gap,Competitive_Intensity,Mean_Panel_Coverage_Pct,Coverage_Status,WCI,WCI_Priority,Annual_Revenue_INR
215,DST_0234,0.302941,0.619858,0.240545,23.128750,UNKNOWN,0.505165,MEDIUM,137538.0
330,DST_0178,0.045588,0.100454,0.007475,49.944167,UNKNOWN,0.254179,LOW,171469.8
294,DST_0232,0.192157,0.589632,0.234114,16.665417,UNKNOWN,0.446019,LOW,274134.6
178,DST_0233,0.367157,0.622012,0.237382,22.207917,UNKNOWN,0.535448,MEDIUM,394075.8
282,DST_0085,0.196078,0.614320,0.234387,19.179167,UNKNOWN,0.456370,LOW,414270.0
309,DST_0229,0.130882,0.606280,0.243285,16.204583,UNKNOWN,0.422438,LOW,434997.0
307,DST_0237,0.120098,0.625311,0.238585,26.195000,UNKNOWN,0.425186,LOW,435774.6
129,DST_0176,0.447549,0.590681,0.245651,16.919583,UNKNOWN,0.559005,MEDIUM,456174.0
186,DST_0238,0.384804,0.582059,0.238704,17.917083,UNKNOWN,0.529142,MEDIUM,462781.8
300,DST_0228,0.192647,0.567253,0.249635,26.805000,UNKNOWN,0.435303,LOW,464088.6


In [97]:
# === PALE DISTRICTS: EXPLICIT REJECTION ===

# Only reliable districts can be explicitly rejected.
reliable_pale = pale_analysis[
    pale_analysis["Coverage_Status"] == "RELIABLE"
].copy()

# Define pale districts using the bottom 25% of annual Kestrel revenue.
pale_revenue_cutoff = reliable_pale["Annual_Revenue_INR"].quantile(0.25)

reliable_pale["Pale_District"] = (
    reliable_pale["Annual_Revenue_INR"] <= pale_revenue_cutoff
)

# Explicit rejection:
# - pale sales
# - reliable panel evidence
# - WCI below the assessment recommendation threshold
rejected_pale_districts = reliable_pale[
    (reliable_pale["Pale_District"]) &
    (reliable_pale["WCI"] < 0.70)
].copy()

# Add business interpretation
rejected_pale_districts["Rejection_Reason"] = np.select(
    [
        rejected_pale_districts["Demand_Signal"] < 0.50,
        rejected_pale_districts["Mean_Distribution_Gap"] < 0.40
    ],
    [
        "Weak independent demand; marketing expansion not justified",
        "Limited distribution gap; insufficient whitespace opportunity"
    ],
    default="Low overall whitespace opportunity"
)

rejected_pale_districts = (
    rejected_pale_districts
    .sort_values("Annual_Revenue_INR")
    .reset_index(drop=True)
)

print("=== REJECTED PALE DISTRICTS ===")
print("Reliable districts:", len(reliable_pale))
print("Pale revenue cutoff (25th percentile):",
      round(pale_revenue_cutoff, 2))
print("Pale districts:",
      reliable_pale["Pale_District"].sum())
print("Explicitly rejected pale districts:",
      len(rejected_pale_districts))

display(
    rejected_pale_districts[
        [
            "district_id",
            "Demand_Signal",
            "Mean_Distribution_Gap",
            "Competitive_Intensity",
            "Mean_Panel_Coverage_Pct",
            "WCI",
            "WCI_Priority",
            "Annual_Revenue_INR",
            "Rejection_Reason"
        ]
    ].head(30)
)

=== REJECTED PALE DISTRICTS ===
Reliable districts: 83
Pale revenue cutoff (25th percentile): 9730804.5
Pale districts: 21
Explicitly rejected pale districts: 7


,district_id,Demand_Signal,Mean_Distribution_Gap,Competitive_Intensity,Mean_Panel_Coverage_Pct,WCI,WCI_Priority,Annual_Revenue_INR,Rejection_Reason
0,DST_0321,0.182353,0.143708,0.008105,60.980000,0.330736,LOW,711217.8,Weak independent demand; marketing expansion n...
1,DST_0061,0.018137,0.104043,0.017332,64.840417,0.241111,LOW,1159232.4,Weak independent demand; marketing expansion n...
2,DST_0174,0.034314,0.155189,0.013496,60.444167,0.267058,LOW,1611122.4,Weak independent demand; marketing expansion n...
3,DST_0203,0.015196,0.117482,0.016830,60.317500,0.244591,LOW,2918215.8,Weak independent demand; marketing expansion n...
4,DST_0129,0.020098,0.087928,0.014226,62.266250,0.236974,LOW,3387036.6,Weak independent demand; marketing expansion n...
5,DST_0164,0.273039,0.425812,0.244422,62.204583,0.423018,LOW,9596230.2,Weak independent demand; marketing expansion n...
6,DST_0183,0.648039,0.427066,0.238828,64.931667,0.593325,MEDIUM,9677473.2,Low overall whitespace opportunity
